
# LeafCareNet — Final 8-Model Journal Benchmark (Dynamic 2×T4 + Resume Bundles)

This notebook retrains the final controlled benchmark on the verified source-clean split using **no additional online augmentation**.

### Final model set

| ID | Model | Role |
|---|---|---|
| 01 | MobileNetV2 Flat | Same-backbone flat baseline |
| 02 | EfficientNet-B3 Flat | CNN baseline |
| 03 | FasterNet-M Flat | CNN baseline |
| 04 | DeiT-Tiny Flat | Lightweight transformer baseline |
| 05 | **LeafCareNet (Proposed)** | **Linear Hierarchical MobileNetV2** |
| 06 | EfficientNet-B3 Hierarchical | Cross-backbone linear hierarchical CNN |
| 07 | FasterNet-M Hierarchical | Cross-backbone linear hierarchical CNN |
| 08 | DeiT-Tiny Hierarchical | Linear hierarchical transformer |

The removed models are **FasterNet-M Hierarchical + SE** and the **old FasterNet-based LeafCareNet**.

### Final LeafCareNet architecture — previous linear hierarchical MobileNetV2

The proposed **LeafCareNet keeps the exact earlier linear Hierarchical MobileNetV2 architecture**. No SE attention, hidden FC-256 layer, or dropout is added to the hierarchical heads.

For MobileNetV2, the ImageNet-pretrained backbone produces the shared feature vector, then:

- **Crop router:** `shared feature → Linear(feature_dim, 3)`  
- **Potato disease head:** `shared feature → Linear(feature_dim, 3)`  
- **Rice disease head:** `shared feature → Linear(feature_dim, 5)`  
- **Tomato disease head:** `shared feature → Linear(feature_dim, 8)`  
- Softmax is used only during inference/evaluation; training uses raw logits with cross-entropy.
- During training, only the disease head corresponding to each sample's true crop contributes to the disease loss.
- During true end-to-end inference, the **predicted crop** selects the crop-specific disease head.

The **same linear hierarchical design** is also applied to EfficientNet-B3, FasterNet-M, and DeiT-Tiny. This gives a clean backbone-matched experiment: each backbone has one flat classifier and one linear hierarchical classifier.

### Fast two-GPU scheduling

This is **not fixed pairing**. GPU 0 and GPU 1 each train one independent model. As soon as either GPU finishes its current model, the next pending model starts immediately on that free GPU.

### Session-loss / new-session resume

After **every completed model**, the notebook immediately creates a full ZIP containing its checkpoint, predictions, reports, figures, logs, configuration, worker code, and completion marker. In a later Kaggle session, attach the clean dataset **plus the completed model ZIP files** as Kaggle inputs and rerun this notebook. It automatically restores completed model bundles and skips them, continuing with the next pending experiment.

Resume granularity is **completed model**. If a session dies halfway through one model, that unfinished model must restart unless Kaggle persistence preserved `/kaggle/working`.


### Reuse of previously trained legacy ZIP files

This version can also reuse the older result ZIP files produced by the previous notebooks.  
The ZIP filename itself does **not** need to follow the new `JOURNAL8_*` naming.

When old ZIPs are attached under `/kaggle/input`, the notebook scans their internal
`result.json`, predictions, checkpoint, and completion metadata. Compatible legacy
experiments are mapped automatically:

- `01_flat_mobilenetv2` → MobileNetV2 Flat
- `02_flat_efficientnet_b3` → EfficientNet-B3 Flat
- `03_flat_fasternet_m` → FasterNet-M Flat
- `09_hier_mobilenetv2_linear` → **LeafCareNet (Proposed)**
- `10_hier_efficientnet_b3_linear` → EfficientNet-B3 Hierarchical
- `06_hier_fasternet_m_linear` → FasterNet-M Hierarchical

The old `07_hier_fasternet_m_se` and old FasterNet-based
`08_leafcarenet_full` are intentionally ignored because they are not part of
the final eight-model benchmark.

A legacy model is skipped only after the notebook validates its architecture/result
schema and finds the held-out test predictions for all **916** test images. When
available, checkpoint/completion/history files are also restored. Among duplicate
copies, the most complete candidate is selected.


### Reuse from the current `/kaggle/working` session

This version also scans compatible raw result folders already present under
`/kaggle/working`. Therefore, if the old completed models are still visible in
the current Kaggle draft session, they can be verified and skipped even when
no old ZIP was attached through `/kaggle/input`.


### GPU-feeding optimization in this version

When DeiT-Tiny still needs training, the notebook builds one shared deterministic
uint8 preprocessing cache for the verified 18,296 source-clean images. The two
independent DeiT workers memory-map this same cache. Repeated JPEG decoding,
resize/crop, and CPU normalization are therefore removed from every epoch;
normalization is performed on each T4. The global training batch remains 128,
so the training protocol is not changed.

This substantially reduces the CPU input bottleneck and should keep the GPUs
much busier. Exact 100% GPU utilization cannot be guaranteed because validation,
optimizer steps, Python scheduling, and model characteristics also affect the
instantaneous utilization reading.

At the end, the notebook creates a single:
`LeafCareNet_JOURNAL8_ALL_8_MODELS_COMPLETE.zip`
containing all eight full model bundles and the complete journal-analysis output.


### Confusion-matrix color correction

All confusion matrices for all eight models are forcibly regenerated from the saved raw
`.npy` matrices with Matplotlib `cmap="Blues"` before the final analysis/model ZIPs are
created. This also replaces legacy Viridis-colored confusion matrices restored from older
runs. No retraining or metric recomputation is involved.


In [5]:

# ============================================================
# 1. GLOBAL SETUP / REGISTRY
# ============================================================

import os
import sys
import gc
import json
import time
import shutil
import zipfile
import threading
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

BASE_OUTPUT = Path(
    "/kaggle/working/leafcarenet_journal8_dynamic"
)

MODEL_ZIP_DIR = Path(
    "/kaggle/working/leafcarenet_journal8_model_zips"
)

WORKER_SCRIPT_PATH = Path(
    "/kaggle/working/leafcarenet_journal8_worker.py"
)

# Shared deterministic preprocessing cache used by the pending DeiT models.
# Images are spatially preprocessed once and stored as uint8 (~2.6 GiB for
# train+val+test), then normalized on the GPU during training.
SPEED_CACHE_ROOT = Path(
    "/kaggle/working/leafcarenet_fast_uint8_cache"
)

BASE_OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_ZIP_DIR.mkdir(parents=True, exist_ok=True)

# Previous experiments showed BS256 was not a stable common single-T4 batch.
# Use the already validated fair common batch directly to avoid wasting time.
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 512
PREFETCH_FACTOR = 8

HOST_CPU_COUNT = os.cpu_count() or 4
PER_MODEL_NUM_WORKERS = max(1, min(2, HOST_CPU_COUNT // 2))

MODEL_REGISTRY = [
    {
        "experiment": "01_flat_mobilenetv2",
        "display_name": "MobileNetV2 Flat",
        "role": "Same-backbone flat baseline",
        "backbone": "mobilenetv2",
        "family": "flat",
    },
    {
        "experiment": "02_flat_efficientnet_b3",
        "display_name": "EfficientNet-B3 Flat",
        "role": "CNN baseline",
        "backbone": "efficientnet_b3",
        "family": "flat",
    },
    {
        "experiment": "03_flat_fasternet_m",
        "display_name": "FasterNet-M Flat",
        "role": "CNN baseline",
        "backbone": "fasternet_m",
        "family": "flat",
    },
    {
        "experiment": "04_flat_deit_tiny",
        "display_name": "DeiT-Tiny Flat",
        "role": "Transformer baseline",
        "backbone": "deit_tiny",
        "family": "flat",
    },
    {
        "experiment": "05_leafcarenet",
        "display_name": "LeafCareNet (Proposed)",
        "role": "Hierarchical MobileNetV2",
        "backbone": "mobilenetv2",
        "family": "hierarchical",
    },
    {
        "experiment": "06_hier_efficientnet_b3",
        "display_name": "EfficientNet-B3 Hierarchical",
        "role": "Cross-backbone hierarchical CNN",
        "backbone": "efficientnet_b3",
        "family": "hierarchical",
    },
    {
        "experiment": "07_hier_fasternet_m",
        "display_name": "FasterNet-M Hierarchical",
        "role": "Cross-backbone hierarchical CNN",
        "backbone": "fasternet_m",
        "family": "hierarchical",
    },
    {
        "experiment": "08_hier_deit_tiny",
        "display_name": "DeiT-Tiny Hierarchical",
        "role": "Hierarchical transformer",
        "backbone": "deit_tiny",
        "family": "hierarchical",
    },
]

MODEL_META = {
    item["experiment"]: item
    for item in MODEL_REGISTRY
}

EXPERIMENT_ORDER = [
    item["experiment"]
    for item in MODEL_REGISTRY
]

MASTER_STATE = BASE_OUTPUT / "MASTER_STATE.json"

print("=" * 100)
print("LEAFCARENET FINAL 8-MODEL JOURNAL BENCHMARK")
print("=" * 100)
print("Train batch / model :", TRAIN_BATCH_SIZE)
print("Eval batch / model  :", EVAL_BATCH_SIZE)
print("Workers / model     :", PER_MODEL_NUM_WORKERS)
print("Scheduler           : dynamic 2-GPU queue")
print("Online augmentation : OFF")
print("Train / Val / Test  : 13,720 / 3,660 / 916")
print("Transformer         : DeiT-Tiny Patch16 224")
print("Hierarchy           : LINEAR crop router + LINEAR crop-specific heads")
print("LeafCareNet         : previous linear Hierarchical MobileNetV2 architecture")
print("=" * 100)


LEAFCARENET FINAL 8-MODEL JOURNAL BENCHMARK
Train batch / model : 128
Eval batch / model  : 512
Workers / model     : 2
Scheduler           : dynamic 2-GPU queue
Online augmentation : OFF
Train / Val / Test  : 13,720 / 3,660 / 916
Transformer         : DeiT-Tiny Patch16 224
Hierarchy           : LINEAR crop router + LINEAR crop-specific heads
LeafCareNet         : previous linear Hierarchical MobileNetV2 architecture


In [6]:
# ============================================================
# 2. WRITE SELF-CONTAINED SINGLE-GPU WORKER
# ============================================================

WORKER_SCRIPT = '\n# ============================================================\n# LEAFCARENET — KAGGLE ONE-CLICK FULL MODEL COMPARISON\n# Dataset: /kaggle/input/datasets/mahbubur03078/leafdisease2\n# GPU: exactly one visible T4 per worker process\n#\n# One run performs:\n#   1) Minimal fresh-session path/GPU/model-count checks\n#   2) 6 controlled model experiments\n#   3) Accuracy / Macro-F1 / Weighted-F1 evaluation\n#   4) Stage-1, Stage-2, true end-to-end evaluation\n#   5) LeafCareNet ablation comparison\n#   6) All required 300-DPI publication PNG figures\n#   7) CSV/JSON/Markdown result tables\n#   8) Best model checkpoints\n#   9) Final ZIP archive in /kaggle/working\n# ============================================================\n\nimport os\nimport sys\nimport re\nimport gc\nimport json\nimport math\nimport time\nimport random\nimport shutil\nimport hashlib\nimport platform\nimport subprocess\nimport warnings\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\n\nwarnings.filterwarnings("ignore")\n\n# Keep CPU libraries from spawning nested thread pools inside each DataLoader worker.\n# The image pipeline is parallelized by DataLoader processes instead.\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nos.environ.setdefault("NUMEXPR_NUM_THREADS", "1")\nos.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")\n\n\n# ---------------------------\n# 0. Install/import packages\n# ---------------------------\ndef ensure_package(import_name, pip_name=None):\n    try:\n        __import__(import_name)\n    except ImportError:\n        pkg = pip_name or import_name\n        subprocess.check_call(\n            [sys.executable, "-m", "pip", "install", "-q", pkg]\n        )\n\nensure_package("timm")\nensure_package("sklearn", "scikit-learn")\nensure_package("pandas")\nensure_package("PIL", "Pillow")\nensure_package("matplotlib")\nensure_package("tqdm")\nensure_package("scipy")\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\nimport matplotlib.pyplot as plt\n\n# Be robust to partially truncated internet-downloaded files that PIL can still decode.\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\n\nimport timm\nfrom timm.data import create_transform, resolve_model_data_config\n\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    precision_recall_fscore_support,\n    classification_report,\n    confusion_matrix,\n    matthews_corrcoef,\n    cohen_kappa_score,\n    f1_score,\n)\nfrom scipy.stats import binomtest\n\nfrom tqdm.auto import tqdm\n\n\n\n\n# ============================================================\n# 1. CONFIGURATION\n# ============================================================\n\nKAGGLE_DATASET_ROOT = Path("/kaggle/input/datasets/mahbubur03078/leafdisease2")\nDATA_ROOT = KAGGLE_DATASET_ROOT\n# Compatibility aliases; no in-notebook cleaning/re-splitting is performed.\nRAW_DATA_ROOT = KAGGLE_DATASET_ROOT\nCLEAN_DATA_ROOT = KAGGLE_DATASET_ROOT\nOUTPUT_ROOT = Path(os.environ["LEAFCARENET_OUTPUT_ROOT"])\nFIG_ROOT = OUTPUT_ROOT / "figures_300dpi"\nTABLE_ROOT = OUTPUT_ROOT / "tables"\nCHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"\nPREDICTION_ROOT = OUTPUT_ROOT / "predictions"\nAUDIT_ROOT = OUTPUT_ROOT / "dataset_audit"\nLOG_ROOT = OUTPUT_ROOT / "logs"\n\nfor p in [\n    OUTPUT_ROOT,\n    FIG_ROOT,\n    TABLE_ROOT,\n    CHECKPOINT_ROOT,\n    PREDICTION_ROOT,\n    AUDIT_ROOT,\n    LOG_ROOT,\n]:\n    p.mkdir(parents=True, exist_ok=True)\n\n# Full console + tqdm log. Output remains visible in Kaggle and is also saved.\nclass TeeStream:\n    def __init__(self, *streams):\n        self.streams = streams\n\n    def write(self, data):\n        for stream in self.streams:\n            stream.write(data)\n        return len(data)\n\n    def flush(self):\n        for stream in self.streams:\n            stream.flush()\n\n    def isatty(self):\n        return False\n\n    @property\n    def encoding(self):\n        return getattr(self.streams[0], "encoding", "utf-8")\n\n_ORIGINAL_STDOUT = sys.stdout\n_ORIGINAL_STDERR = sys.stderr\n_LOG_FILE = open(\n    LOG_ROOT / "training_run.log",\n    "a",\n    buffering=1,\n    encoding="utf-8",\n)\nsys.stdout = TeeStream(_ORIGINAL_STDOUT, _LOG_FILE)\nsys.stderr = TeeStream(_ORIGINAL_STDERR, _LOG_FILE)\n\nRUN_STARTED_AT = time.strftime("%Y-%m-%d %H:%M:%S")\nprint("=" * 100)\nprint("FINAL JOURNAL RUN LOG STARTED:", RUN_STARTED_AT)\nprint("Log file:", LOG_ROOT / "training_run.log")\nprint("=" * 100)\n\nSEED = 42\nIMG_SIZE = 224\nBATCH_SIZE = int(os.environ.get("LEAFCARENET_BATCH_SIZE", "256"))\nEVAL_BATCH_SIZE = int(os.environ.get("LEAFCARENET_EVAL_BATCH_SIZE", "512"))\n\n# Optional shared deterministic preprocessing cache.\n# The cache stores spatially preprocessed 224x224 RGB tensors as uint8.\n# Normalization is performed on the GPU, eliminating repeated JPEG decode,\n# resize/crop, ToTensor, and CPU normalization every epoch.\n_UINT8_CACHE_ENV = os.environ.get("LEAFCARENET_UINT8_CACHE_DIR", "").strip()\nUINT8_CACHE_DIR = Path(_UINT8_CACHE_ENV) if _UINT8_CACHE_ENV else None\nUSE_PREPROCESSED_UINT8_CACHE = (\n    UINT8_CACHE_DIR is not None\n    and (UINT8_CACHE_DIR / "cache_meta.json").exists()\n)\n\nACTIVE_INPUT_MEAN = None\nACTIVE_INPUT_STD = None\n_GPU_NORM_MEAN = None\n_GPU_NORM_STD = None\n\n# Original LeafCareNet training protocol\nMAX_EPOCHS = 100\nPATIENCE = 5\nMIN_DELTA = 1e-4\nLR = 1e-3\nWEIGHT_DECAY = 1e-4\n\nCROP_LOSS_WEIGHT = 2.0\nCROP_LABEL_SMOOTHING = 0.1\nFLAT_LABEL_SMOOTHING = 0.1\n\nCPU_COUNT = os.cpu_count() or 4\n\n# Two independent single-GPU model processes run concurrently.\nNUM_WORKERS = int(\n    os.environ.get(\n        "LEAFCARENET_NUM_WORKERS",\n        str(max(1, min(4, CPU_COUNT // 2))),\n    )\n)\nAUTO_TUNE_NUM_WORKERS = False\nWORKER_TUNE_CANDIDATES = [NUM_WORKERS]\nWORKER_TUNE_WARMUP_BATCHES = 0\nWORKER_TUNE_TIMED_BATCHES = 0\n\nPREFETCH_FACTOR = int(\n    os.environ.get("LEAFCARENET_PREFETCH_FACTOR", "3")\n)\n\nTQDM_MININTERVAL = 2.0\nUSE_CHANNELS_LAST = True\nUSE_DIRECT_DP_CPU_SCATTER = False\nUSE_FUSED_ADAMW = True\n\n# No decoded-RGB memmap in this parallel version.\nUSE_RGB_MEMMAP_CACHE = False\nRGB_CACHE_MAX_GIB = 0.0\nRGB_CACHE_REBUILD = False\nUSE_AMP = True\nPRETRAINED = os.environ.get("LEAFCARENET_PREFLIGHT", "0") != "1"\nBOOTSTRAP_REPEATS = 1000\nFAIL_ON_EXACT_CROSS_SPLIT_DUPLICATES = True\nLATENCY_WARMUP = 20\nLATENCY_REPEATS = 100\n\n# Uploaded dataset provenance / validation settings\nPREBUILT_SOURCE_CLEAN_DATASET = True\n\n# The Kaggle upload physically contains train-only offline augmented copies,\n# but this final run intentionally excludes them from every training Dataset.\nUPLOADED_OFFLINE_AUGMENTATION_PRESENT = True\nUSE_OFFLINE_AUGMENTED_FILES_FOR_TRAINING = False\n\n# Restore the original LeafCareNet training-time augmentation protocol.\nONLINE_AUGMENTATION = False\nOFFLINE_AUGMENTATION = False\n\nSOURCE_SPLIT_RATIOS = {"train": 0.75, "val": 0.20, "test": 0.05}\nEXTERNAL_NEAR_DUPLICATE_THRESHOLD = 4\n\n# Physical uploaded dataset counts.\nEXPECTED_UPLOADED_SPLIT_COUNTS = {"train": 68600, "val": 3660, "test": 916}\nEXPECTED_UPLOADED_TRAIN_ORIGINALS = 13720\nEXPECTED_UPLOADED_TRAIN_AUGMENTED = 54880\nEXPECTED_UPLOADED_TOTAL_IMAGES = 73176\n\n# Files actually used by the eight model experiments.\nEXPECTED_MODEL_SPLIT_COUNTS = {"train": 13720, "val": 3660, "test": 916}\nEXPECTED_MODEL_TOTAL_IMAGES = 18296\n\n\n# Eight final journal models: four backbone-matched flat/hierarchical comparisons.\n# The SAME LINEAR hierarchy is transferred across backbones for controlled analysis.\n# The MobileNetV2 hierarchical instance is the proposed LeafCareNet.\n# LeafCareNet therefore keeps the exact earlier architecture that produced the\n# previous Hierarchical MobileNetV2 result: Linear crop router + linear crop-specific heads.\nEXPERIMENTS = [\n    ("01_flat_mobilenetv2", "flat", "mobilenetv2", "flat"),\n    ("02_flat_efficientnet_b3", "flat", "efficientnet_b3", "flat"),\n    ("03_flat_fasternet_m", "flat", "fasternet_m", "flat"),\n    ("04_flat_deit_tiny", "flat", "deit_tiny", "flat"),\n\n    ("05_leafcarenet", "hierarchical", "mobilenetv2", "linear"),\n    ("06_hier_efficientnet_b3", "hierarchical", "efficientnet_b3", "linear"),\n    ("07_hier_fasternet_m", "hierarchical", "fasternet_m", "linear"),\n    ("08_hier_deit_tiny", "hierarchical", "deit_tiny", "linear"),\n]\n\nCROPS = ["potato", "rice", "tomato"]\nSPLITS = ["train", "val", "test"]\nIMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n\nBACKBONE_CANDIDATES = {\n    "mobilenetv2": [\n        "mobilenetv2_100.ra_in1k",\n        "mobilenetv2_100",\n    ],\n    "efficientnet_b3": [\n        "efficientnet_b3.ra2_in1k",\n        "efficientnet_b3",\n    ],\n    "fasternet_m": [\n        "fasternet_m.in1k",\n        "fasternet_m",\n    ],\n    # Lightweight transformer baseline chosen for fast T4 training.\n    "deit_tiny": [\n        "deit_tiny_patch16_224.fb_in1k",\n        "deit_tiny_patch16_224",\n    ],\n}\n\n\n\n\n# ============================================================\n# 2. REPRODUCIBILITY / GPU\n# ============================================================\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n\n    # Deterministic where practical; benchmark improves T4 throughput.\n    torch.backends.cudnn.benchmark = True\n\nseed_everything(SEED)\n\n\n# Keep the main process lightweight; DataLoader worker processes own image preprocessing.\ntry:\n    torch.set_num_threads(1)\nexcept Exception:\n    pass\n\ntry:\n    torch.set_num_interop_threads(1)\nexcept Exception:\n    pass\n\n\n\n# Execution-only CUDA optimizations.\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\n\ntry:\n    torch.set_float32_matmul_precision("high")\nexcept Exception:\n    pass\n\nprint("=" * 90)\nprint("LEAFCARENET FINAL JOURNAL ONE-CLICK COMPARISON")\nprint("=" * 90)\nprint("Python:", sys.version.split()[0])\nprint("PyTorch:", torch.__version__)\nprint("timm:", timm.__version__)\nprint("CUDA available:", torch.cuda.is_available())\nprint("GPU count:", torch.cuda.device_count())\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\n        "GPU is not enabled. Kaggle: Settings > Accelerator > GPU."\n    )\n\nfor i in range(torch.cuda.device_count()):\n    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")\n\nDEVICE = torch.device("cuda:0")\nUSE_MULTI_GPU = torch.cuda.device_count() > 1\ntorch.cuda.set_device(0)\nprint("Visible-GPU DataParallel enabled:", USE_MULTI_GPU)\nprint("Detected CPU logical cores:", CPU_COUNT)\nprint("Initial DataLoader workers:", NUM_WORKERS)\nprint("Worker autotune candidates:", WORKER_TUNE_CANDIDATES)\nprint("Prefetch factor:", PREFETCH_FACTOR)\nprint("Channels-last:", USE_CHANNELS_LAST)\nprint("Train batch size:", BATCH_SIZE)\nprint("Eval batch size:", EVAL_BATCH_SIZE)\nprint("Direct DP CPU scatter:", USE_DIRECT_DP_CPU_SCATTER and USE_MULTI_GPU)\nprint(\n    "Preprocessed uint8 cache:",\n    str(UINT8_CACHE_DIR) if USE_PREPROCESSED_UINT8_CACHE else "DISABLED",\n)\n\n\n\n\nif torch.cuda.device_count() != 1:\n    raise RuntimeError(\n        "Each parallel worker must see exactly one GPU. "\n        f"Visible GPU count: {torch.cuda.device_count()}"\n    )\n\n\n# ============================================================\n# 3. DATASET PATH AUTO-CHECK\n# ============================================================\n\ndef has_expected_crop_structure(root):\n    root = Path(root)\n\n    return all(\n        (root / f"{crop}_leaf").is_dir()\n        for crop in CROPS\n    )\n\n\ndef locate_dataset():\n    """\n    Kaggle datasets sometimes preserve the uploaded top-level folder.\n    Therefore the configured leafdisease2 root is searched recursively\n    for the directory that directly contains potato_leaf/rice_leaf/tomato_leaf.\n    """\n    global DATA_ROOT, RAW_DATA_ROOT, CLEAN_DATA_ROOT\n\n    configured = Path(KAGGLE_DATASET_ROOT)\n\n    if not configured.exists():\n        raise FileNotFoundError(\n            f"Configured Kaggle dataset does not exist: {configured}"\n        )\n\n    if has_expected_crop_structure(configured):\n        resolved = configured\n    else:\n        candidates = []\n\n        for potato_dir in configured.rglob("potato_leaf"):\n            parent = potato_dir.parent\n\n            if has_expected_crop_structure(parent):\n                candidates.append(parent)\n\n        # Fallback search through all Kaggle inputs if the dataset slug changes.\n        if not candidates:\n            base = Path("/kaggle/input")\n\n            for potato_dir in base.rglob("potato_leaf"):\n                parent = potato_dir.parent\n\n                if has_expected_crop_structure(parent):\n                    candidates.append(parent)\n\n        if not candidates:\n            raise FileNotFoundError(\n                "Could not locate a directory containing potato_leaf, "\n                "rice_leaf and tomato_leaf."\n            )\n\n        # Prefer candidates physically inside leafdisease2.\n        inside_configured = [\n            p for p in candidates\n            if configured == p or configured in p.parents\n        ]\n\n        resolved = (\n            sorted(inside_configured, key=lambda p: len(p.parts))[0]\n            if inside_configured\n            else sorted(candidates, key=lambda p: len(p.parts))[0]\n        )\n\n    DATA_ROOT = resolved\n    RAW_DATA_ROOT = resolved\n    CLEAN_DATA_ROOT = resolved\n\n    print("Configured Kaggle dataset root:", configured)\n    print("Resolved training dataset root:", DATA_ROOT)\n\n    return DATA_ROOT\n\n\nlocate_dataset()\n\n# ------------------------------------------------------------\n# Final experiment file-selection rule\n# ------------------------------------------------------------\ndef is_offline_augmented_filename(name):\n    return "__aug_" in Path(name).stem.lower()\n\n\ndef model_uses_path(path, split):\n    """\n    The Kaggle upload contains offline-generated __aug_* files in train.\n    They are excluded from the final model Dataset objects.\n    Validation/test are used as prepared.\n    """\n    path = Path(path)\n\n    if split == "train" and is_offline_augmented_filename(path.name):\n        return False\n\n    return True\n\n\n\n# ============================================================\n# 4. DIRECT COMPRESSED-IMAGE LOADING — NO FULL RGB MEMMAP\n# ============================================================\n#\n# Why:\n#   The previous full-resolution RGB cache occupied ~11+ GiB and caused heavy\n#   random memory-mapped I/O. This V2 loader reads the original compressed\n#   JPG/PNG/WebP files directly and lets persistent DataLoader workers perform\n#   decode + online augmentation in parallel.\n#\n# Methodology:\n#   image file -> PIL RGB decode -> SAME online train transform\n#   validation/test -> deterministic eval transform\n# ============================================================\n\nRGB_CACHE_DIR = OUTPUT_ROOT / "rgb_decode_cache"\nRGB_CACHE_BIN = RGB_CACHE_DIR / "disabled.bin"\nRGB_CACHE_INDEX_CSV = RGB_CACHE_DIR / "disabled.csv"\nRGB_CACHE_INDEX = {}\nRGB_CACHE_ACTIVE = False\n\n# Free only legacy execution-cache folders from earlier notebook versions.\n# Checkpoints/tables/results are NOT removed here.\nLEGACY_CACHE_DIRS = [\n    Path(\n        "/kaggle/working/"\n        "leafcarenet_final_original_onlineaug_journal_run/"\n        "rgb_decode_cache"\n    ),\n]\n\nfor legacy_cache_dir in LEGACY_CACHE_DIRS:\n    if legacy_cache_dir.exists():\n        try:\n            shutil.rmtree(legacy_cache_dir)\n            print(\n                "Removed legacy RGB execution cache:",\n                legacy_cache_dir,\n            )\n        except Exception as exc:\n            print(\n                "Could not remove legacy RGB cache:",\n                legacy_cache_dir,\n                exc,\n            )\n\n\ndef load_rgb_image_fast(path):\n    """\n    Direct compressed-file loader.\n\n    The `with` block closes the underlying file immediately after RGB\n    conversion, avoiding accumulation of open file handles in persistent\n    DataLoader workers.\n    """\n    with Image.open(path) as img:\n        return img.convert("RGB")\n\n\nprint("RGB memmap cache: DISABLED")\nprint("Image source: original compressed JPG/PNG/WebP files")\n\n\n# ============================================================\n# 5. BACKBONE HELPERS / TRANSFORMS\n# ============================================================\n\ndef select_backbone(key):\n    # include_tags=True is important: otherwise timm.list_models() can hide\n    # tagged pretrained names such as "fasternet_m.in1k" and silently fall\n    # back to the untagged alias.\n    try:\n        available = set(\n            timm.list_models(\n                include_tags=True\n            )\n        )\n    except TypeError:\n        available = set(\n            timm.list_models()\n        )\n\n    for candidate in BACKBONE_CANDIDATES[key]:\n        if candidate in available:\n            return candidate\n\n    if key == "mobilenetv2":\n        matches = timm.list_models("*mobilenetv2*")\n    elif key == "efficientnet_b3":\n        matches = timm.list_models("*efficientnet_b3*")\n    elif key == "fasternet_m":\n        matches = timm.list_models("*fasternet_m*")\n    elif key == "deit_tiny":\n        matches = timm.list_models("*deit_tiny_patch16_224*")\n    else:\n        matches = []\n\n    if not matches:\n        raise RuntimeError(\n            f"No timm model found for {key}"\n        )\n\n    return matches[0]\n\n\nRESOLVED_BACKBONES = {\n    key: select_backbone(key) for key in BACKBONE_CANDIDATES\n}\n\nprint("\\nResolved timm backbones:")\nfor k, v in RESOLVED_BACKBONES.items():\n    print(f"  {k}: {v}")\n\n\nclass FasterNetPooledFeatures(nn.Module):\n    """\n    Exact LeafCareNet FasterNet feature interface:\n\n        FasterNet.forward_features(x)\n            -> global spatial average pooling\n            -> 1152-D feature vector for FasterNet-M\n\n    This deliberately bypasses timm FasterNet\'s forward_head / pre-logits\n    expansion. In timm 1.0.26 the unwrapped model(x) path produced 1280-D\n    pre-logits while model.num_features reported the backbone feature width\n    used by LeafCareNet. That mismatch caused:\n        mat1: [B, 1280]\n        head: [1152, ...]\n    """\n\n    def __init__(\n        self,\n        model,\n        feat_dim,\n    ):\n        super().__init__()\n\n        self.model = model\n        self.num_features = int(\n            feat_dim\n        )\n\n        # The task-specific FasterNet experiments use raw backbone features.\n        # Remove any unused timm classification/pre-logits head parameters\n        # when that module is exposed as `head`.\n        if hasattr(\n            self.model,\n            "head",\n        ):\n            self.model.head = nn.Identity()\n\n    def forward(\n        self,\n        x,\n    ):\n        features = (\n            self.model\n            .forward_features(\n                x\n            )\n        )\n\n        if features.ndim == 4:\n            # Normal FasterNet output is NCHW: [B, C, H, W].\n            if (\n                features.shape[1]\n                == self.num_features\n            ):\n                features = (\n                    features\n                    .mean(\n                        dim=(2, 3)\n                    )\n                )\n\n            # Defensive NHWC fallback.\n            elif (\n                features.shape[-1]\n                == self.num_features\n            ):\n                features = (\n                    features\n                    .mean(\n                        dim=(1, 2)\n                    )\n                )\n\n            else:\n                raise RuntimeError(\n                    "Unexpected FasterNet forward_features shape: "\n                    f"{tuple(features.shape)}; expected feature width "\n                    f"{self.num_features}."\n                )\n\n        elif features.ndim == 3:\n            # Defensive token-sequence fallback.\n            features = (\n                features\n                .mean(\n                    dim=1\n                )\n            )\n\n        if (\n            features.ndim != 2\n            or features.shape[1]\n            != self.num_features\n        ):\n            raise RuntimeError(\n                "FasterNet pooled feature mismatch: "\n                f"got {tuple(features.shape)}, "\n                f"expected [batch, {self.num_features}]."\n            )\n\n        return features\n\n\ndef create_backbone(backbone_key):\n    name = RESOLVED_BACKBONES[backbone_key]\n\n    try:\n        model = timm.create_model(\n            name,\n            pretrained=PRETRAINED,\n            num_classes=0,\n            global_pool="avg",\n        )\n    except Exception as exc:\n        raise RuntimeError(\n            f"Could not load pretrained model \'{name}\'. "\n            "For ImageNet pretrained weights, enable Internet in Kaggle. "\n            f"Original error: {exc}"\n        )\n\n    feat_dim = int(\n        model.num_features\n    )\n\n    if backbone_key == "fasternet_m":\n        model = FasterNetPooledFeatures(\n            model,\n            feat_dim,\n        )\n\n        print(\n            "FasterNet feature interface: "\n            f"forward_features -> global mean -> {feat_dim}-D"\n        )\n\n    return model, feat_dim\n\n\ndef get_data_cfg(backbone_key):\n    model, _ = create_backbone(backbone_key)\n\n    # resolve_model_data_config can read through our wrapper\'s underlying\n    # pretrained model config via the wrapped timm model.\n    config_model = (\n        model.model\n        if isinstance(\n            model,\n            FasterNetPooledFeatures,\n        )\n        else model\n    )\n\n    cfg = resolve_model_data_config(\n        config_model\n    )\n\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    return cfg\n\n\ndef make_transforms(backbone_key):\n    global ACTIVE_INPUT_MEAN\n    global ACTIVE_INPUT_STD\n    global _GPU_NORM_MEAN\n    global _GPU_NORM_STD\n\n    cfg = get_data_cfg(backbone_key)\n\n    ACTIVE_INPUT_MEAN = tuple(float(x) for x in cfg["mean"])\n    ACTIVE_INPUT_STD = tuple(float(x) for x in cfg["std"])\n    _GPU_NORM_MEAN = None\n    _GPU_NORM_STD = None\n\n    # FINAL NO-AUGMENTATION PROTOCOL.\n    # Train/val/test all receive deterministic resize/crop/normalization.\n    train_tf = create_transform(\n        input_size=(3, IMG_SIZE, IMG_SIZE),\n        is_training=False,\n        mean=cfg["mean"],\n        std=cfg["std"],\n        interpolation=cfg.get("interpolation", "bicubic"),\n        crop_pct=cfg.get("crop_pct", 0.875),\n    )\n\n    eval_tf = create_transform(\n        input_size=(3, IMG_SIZE, IMG_SIZE),\n        is_training=False,\n        mean=cfg["mean"],\n        std=cfg["std"],\n        interpolation=cfg.get("interpolation", "bicubic"),\n        crop_pct=cfg.get("crop_pct", 0.875),\n    )\n\n    return train_tf, eval_tf\n\n\n# ============================================================\n# 6. DATASETS\n# ============================================================\n\ndef _ordered_sample_hash(samples):\n    """\n    Hash only stable dataset-relative identifiers and class indices.\n    This prevents a cache built in one Kaggle mount path from being accepted\n    with a different ordering or split.\n    """\n    h = hashlib.sha256()\n\n    for (\n        path,\n        crop_idx,\n        disease_idx,\n        global_idx,\n        crop,\n        cls,\n    ) in samples:\n        try:\n            rel = Path(path).resolve().relative_to(\n                DATA_ROOT.resolve()\n            ).as_posix()\n        except Exception:\n            rel = "/".join(\n                Path(path).as_posix().split("/")[-4:]\n            )\n\n        line = (\n            f"{rel}|{crop_idx}|{disease_idx}|"\n            f"{global_idx}|{crop}|{cls}\\n"\n        )\n\n        h.update(\n            line.encode("utf-8")\n        )\n\n    return h.hexdigest()\n\n\nclass HierarchicalLeafDataset(Dataset):\n    def __init__(self, root, split, transform=None):\n        self.root = Path(root)\n        self.split = split\n        self.transform = transform\n        self._uint8_cache = None\n\n        self.crop_to_idx = {crop: i for i, crop in enumerate(CROPS)}\n\n        self.crop_classes = {}\n        self.disease_to_idx = {}\n\n        self.global_classes = []\n        self.global_to_idx = {}\n\n        self.samples = []\n\n        # Discover class names alphabetically, consistent with ImageFolder.\n        for crop in CROPS:\n            split_dir = self.root / f"{crop}_leaf" / split\n\n            classes = sorted(\n                [p.name for p in split_dir.iterdir() if p.is_dir()]\n            )\n\n            self.crop_classes[crop] = classes\n            self.disease_to_idx[crop] = {\n                cls: idx for idx, cls in enumerate(classes)\n            }\n\n        for crop in CROPS:\n            for cls in self.crop_classes[crop]:\n                self.global_to_idx[(crop, cls)] = len(\n                    self.global_classes\n                )\n                self.global_classes.append(f"{crop}::{cls}")\n\n        for crop in CROPS:\n            crop_idx = self.crop_to_idx[crop]\n            split_dir = self.root / f"{crop}_leaf" / split\n\n            for cls in self.crop_classes[crop]:\n                disease_idx = self.disease_to_idx[crop][cls]\n                global_idx = self.global_to_idx[(crop, cls)]\n                cls_dir = split_dir / cls\n\n                for f in sorted(cls_dir.rglob("*")):\n                    if (\n                        f.is_file()\n                        and f.suffix.lower() in IMAGE_EXTS\n                        and model_uses_path(f, split)\n                    ):\n                        self.samples.append(\n                            (\n                                str(f),\n                                crop_idx,\n                                disease_idx,\n                                global_idx,\n                                crop,\n                                cls,\n                            )\n                        )\n\n        # Optional deterministic spatial-preprocessing cache.\n        if USE_PREPROCESSED_UINT8_CACHE:\n            meta_path = UINT8_CACHE_DIR / "cache_meta.json"\n            image_path = UINT8_CACHE_DIR / f"{split}_images.npy"\n\n            if not image_path.exists():\n                raise FileNotFoundError(\n                    f"Preprocessed cache file missing: {image_path}"\n                )\n\n            meta = json.loads(\n                meta_path.read_text(\n                    encoding="utf-8"\n                )\n            )\n\n            split_meta = meta.get(\n                "splits",\n                {}\n            ).get(\n                split\n            )\n\n            if split_meta is None:\n                raise RuntimeError(\n                    f"Cache metadata has no split \'{split}\'."\n                )\n\n            expected_hash = _ordered_sample_hash(\n                self.samples\n            )\n\n            if int(split_meta.get("count", -1)) != len(self.samples):\n                raise RuntimeError(\n                    f"Cache count mismatch for {split}: "\n                    f"cache={split_meta.get(\'count\')} "\n                    f"dataset={len(self.samples)}"\n                )\n\n            if split_meta.get("sample_hash") != expected_hash:\n                raise RuntimeError(\n                    f"Cache sample/order hash mismatch for {split}."\n                )\n\n            self._uint8_cache = np.load(\n                image_path,\n                mmap_mode="r",\n            )\n\n            expected_shape = (\n                len(self.samples),\n                3,\n                IMG_SIZE,\n                IMG_SIZE,\n            )\n\n            if tuple(self._uint8_cache.shape) != expected_shape:\n                raise RuntimeError(\n                    f"Cache tensor shape mismatch for {split}: "\n                    f"{self._uint8_cache.shape} != {expected_shape}"\n                )\n\n            if self._uint8_cache.dtype != np.uint8:\n                raise RuntimeError(\n                    f"Cache dtype must be uint8, got {self._uint8_cache.dtype}"\n                )\n\n            print(\n                f"FAST CACHE ACTIVE [{split}]: "\n                f"{len(self.samples)} images | "\n                f"{image_path.name}"\n            )\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        (\n            path,\n            crop_idx,\n            disease_idx,\n            global_idx,\n            crop,\n            cls,\n        ) = self.samples[idx]\n\n        if self._uint8_cache is not None:\n            # Copy one ~150 KB uint8 sample from the memory map.\n            # This is dramatically cheaper than JPEG decode + resize/crop.\n            image = torch.from_numpy(\n                np.array(\n                    self._uint8_cache[idx],\n                    copy=True,\n                )\n            )\n\n        else:\n            image = load_rgb_image_fast(path)\n\n            if self.transform:\n                image = self.transform(image)\n\n        return image, crop_idx, disease_idx, global_idx\n\n\nclass FlatLeafDataset(Dataset):\n    def __init__(self, hierarchical_dataset):\n        self.base = hierarchical_dataset\n        self.samples = hierarchical_dataset.samples\n        self.global_classes = hierarchical_dataset.global_classes\n\n    def __len__(self):\n        return len(self.base)\n\n    def __getitem__(self, idx):\n        image, _, _, global_idx = self.base[idx]\n        return image, global_idx\n\n\n\n\n# ============================================================\n# 7. BALANCED SAMPLERS\n# ============================================================\n\ndef build_hierarchical_balanced_sampler(dataset):\n    counts = Counter()\n\n    for _, crop_idx, disease_idx, _, _, _ in dataset.samples:\n        counts[(crop_idx, disease_idx)] += 1\n\n    weights = []\n\n    for _, crop_idx, disease_idx, _, _, _ in dataset.samples:\n        # Equivalent proportional form of:\n        # (1 / n_crops) * (1 / disease_count_within_crop)\n        w = (1.0 / len(CROPS)) * (\n            1.0 / counts[(crop_idx, disease_idx)]\n        )\n        weights.append(w)\n\n    return WeightedRandomSampler(\n        torch.DoubleTensor(weights),\n        num_samples=len(weights),\n        replacement=True,\n    )\n\n\ndef build_flat_balanced_sampler(dataset):\n    counts = Counter(\n        global_idx\n        for _, _, _, global_idx, _, _ in dataset.base.samples\n    )\n\n    weights = [\n        1.0 / counts[global_idx]\n        for _, _, _, global_idx, _, _ in dataset.base.samples\n    ]\n\n    return WeightedRandomSampler(\n        torch.DoubleTensor(weights),\n        num_samples=len(weights),\n        replacement=True,\n    )\n\n\n\n\n# ============================================================\n# 8. MODEL DEFINITIONS\n# ============================================================\n\nclass SEGate(nn.Module):\n    def __init__(self, dim, reduction=16):\n        super().__init__()\n\n        hidden = max(dim // reduction, 8)\n\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, dim)\n\n    def forward(self, x):\n        attn = torch.relu(self.fc1(x))\n        attn = torch.sigmoid(self.fc2(attn))\n        return x * attn\n\n\nclass LinearDiseaseHead(nn.Module):\n    def __init__(self, dim, num_classes):\n        super().__init__()\n        self.fc = nn.Linear(dim, num_classes)\n\n    def forward(self, x):\n        return self.fc(x)\n\n\nclass JournalDiseaseHead(nn.Module):\n    """\n    Crop-specific disease head used in the final journal hierarchy:\n        feature -> FC(256) -> ReLU -> Dropout(0.5) -> class logits\n    """\n    def __init__(self, dim, num_classes):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(dim, 256),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.50),\n            nn.Linear(256, num_classes),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass JournalCropRouter(nn.Module):\n    """\n    Final LeafCareNet crop router matching the architecture figure:\n        SE(r=8) -> FC(256) -> ReLU -> Dropout(0.5) -> 3 crop logits.\n\n    Softmax is intentionally NOT inside the module because CrossEntropyLoss\n    requires raw logits. Softmax is applied only at evaluation/inference.\n    """\n    def __init__(self, dim, n_crops=3, reduction=8):\n        super().__init__()\n        self.se = SEGate(dim, reduction=reduction)\n        self.net = nn.Sequential(\n            nn.Linear(dim, 256),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.50),\n            nn.Linear(256, n_crops),\n        )\n\n    def forward(self, x):\n        return self.net(self.se(x))\n\n\nclass SEDiseaseHead(nn.Module):\n    def __init__(self, dim, num_classes, reduction=16):\n        super().__init__()\n        self.se = SEGate(dim, reduction)\n        self.fc = nn.Linear(dim, num_classes)\n\n    def forward(self, x):\n        return self.fc(self.se(x))\n\n\nclass DeepCropRouter(nn.Module):\n    """\n    Full custom LeafCareNet crop router:\n    SE(r=8) -> 1152 -> 512 -> 256 -> 3\n    BatchNorm + GELU + Dropout.\n    """\n\n    def __init__(self, dim, n_crops=3, reduction=8):\n        super().__init__()\n\n        self.se = SEGate(dim, reduction)\n\n        self.net = nn.Sequential(\n            nn.Linear(dim, 512),\n            nn.BatchNorm1d(512),\n            nn.GELU(),\n            nn.Dropout(0.30),\n\n            nn.Linear(512, 256),\n            nn.BatchNorm1d(256),\n            nn.GELU(),\n            nn.Dropout(0.15),\n\n            nn.Linear(256, n_crops),\n        )\n\n    def forward(self, x):\n        return self.net(self.se(x))\n\n\nclass FlatClassifier(nn.Module):\n    def __init__(self, backbone_key, n_classes=16):\n        super().__init__()\n\n        self.backbone, feat_dim = create_backbone(backbone_key)\n        self.feat_dim = feat_dim\n        self.head = nn.Linear(feat_dim, n_classes)\n\n    def forward(self, x):\n        # DataParallel first scatters NCHW chunks to each GPU.\n        # Convert each chunk to channels-last on its own GPU, avoiding a\n        # large CPU memory-format copy of the full batch.\n        if USE_CHANNELS_LAST and x.is_cuda:\n            x = x.contiguous(\n                memory_format=torch.channels_last\n            )\n\n        features = self.backbone(x)\n        return self.head(features)\n\n\nclass HierarchicalClassifier(nn.Module):\n    """\n    variant = linear:\n        linear crop router + linear disease heads\n\n    variant = se:\n        shallow SE crop router + SE disease heads\n\n    variant = leafcarenet:\n        legacy FasterNet LeafCareNet variant (kept only for code compatibility)\n\n    NOTE:\n        The final journal benchmark uses variant="linear" for LeafCareNet\n        (Hierarchical MobileNetV2) and for all cross-backbone hierarchical\n        controls. This preserves the exact earlier linear hierarchical\n        MobileNetV2 architecture.\n    """\n\n    def __init__(\n        self,\n        backbone_key,\n        crop_class_counts,\n        variant="linear",\n    ):\n        super().__init__()\n\n        self.variant = variant\n        self.backbone, feat_dim = create_backbone(backbone_key)\n        self.feat_dim = feat_dim\n\n        if variant == "linear":\n            self.crop_router = nn.Linear(feat_dim, 3)\n\n            self.disease_heads = nn.ModuleList(\n                [\n                    LinearDiseaseHead(feat_dim, n)\n                    for n in crop_class_counts\n                ]\n            )\n\n        elif variant == "se":\n            self.crop_router = nn.Sequential(\n                SEGate(feat_dim, reduction=8),\n                nn.Linear(feat_dim, 3),\n            )\n\n            self.disease_heads = nn.ModuleList(\n                [\n                    SEDiseaseHead(\n                        feat_dim,\n                        n,\n                        reduction=16,\n                    )\n                    for n in crop_class_counts\n                ]\n            )\n\n        elif variant == "leafcarenet":\n            self.crop_router = DeepCropRouter(\n                feat_dim,\n                n_crops=3,\n                reduction=8,\n            )\n\n            self.disease_heads = nn.ModuleList(\n                [\n                    SEDiseaseHead(\n                        feat_dim,\n                        n,\n                        reduction=16,\n                    )\n                    for n in crop_class_counts\n                ]\n            )\n\n        elif variant == "journal_hier":  # legacy/unused in final benchmark\n            self.crop_router = JournalCropRouter(\n                feat_dim,\n                n_crops=3,\n                reduction=8,\n            )\n\n            self.disease_heads = nn.ModuleList(\n                [\n                    JournalDiseaseHead(feat_dim, n)\n                    for n in crop_class_counts\n                ]\n            )\n\n        else:\n            raise ValueError(f"Unknown variant: {variant}")\n\n    def forward(self, x):\n        # Perform the memory-format conversion after DataParallel scatter,\n        # independently on each GPU replica.\n        if USE_CHANNELS_LAST and x.is_cuda:\n            x = x.contiguous(\n                memory_format=torch.channels_last\n            )\n\n        features = self.backbone(x)\n\n        crop_logits = self.crop_router(features)\n\n        disease_logits = [\n            head(features) for head in self.disease_heads\n        ]\n\n        return crop_logits, disease_logits\n\n\ndef maybe_parallel(model):\n    model = model.to(DEVICE)\n\n    if USE_CHANNELS_LAST:\n        model = model.to(\n            memory_format=torch.channels_last\n        )\n\n    if USE_MULTI_GPU:\n        model = nn.DataParallel(\n            model,\n            device_ids=list(range(torch.cuda.device_count())),\n            output_device=0,\n            dim=0,\n        )\n\n    return model\n\n\ndef make_fast_adamw(model):\n    """\n    Use fused AdamW on CUDA when the installed PyTorch build supports it.\n    This changes only optimizer implementation, not AdamW hyperparameters.\n    """\n    kwargs = dict(\n        lr=LR,\n        weight_decay=WEIGHT_DECAY,\n    )\n\n    if USE_FUSED_ADAMW:\n        try:\n            opt = torch.optim.AdamW(\n                model.parameters(),\n                fused=True,\n                **kwargs,\n            )\n            print("Optimizer backend: fused AdamW")\n            return opt\n        except (TypeError, RuntimeError):\n            pass\n\n    try:\n        opt = torch.optim.AdamW(\n            model.parameters(),\n            foreach=True,\n            **kwargs,\n        )\n        print("Optimizer backend: foreach AdamW")\n        return opt\n    except TypeError:\n        opt = torch.optim.AdamW(\n            model.parameters(),\n            **kwargs,\n        )\n        print("Optimizer backend: standard AdamW")\n        return opt\n\n\ndef unwrap_model(model):\n    return model.module if isinstance(model, nn.DataParallel) else model\n\n\n\n\n# ============================================================\n# 9. LOSSES\n# ============================================================\n\ncrop_ce = nn.CrossEntropyLoss(\n    label_smoothing=CROP_LABEL_SMOOTHING\n)\ndisease_ce = nn.CrossEntropyLoss()\nflat_ce = nn.CrossEntropyLoss(\n    label_smoothing=FLAT_LABEL_SMOOTHING\n)\n\n\ndef hierarchical_loss(\n    crop_logits,\n    disease_logits,\n    crop_targets,\n    disease_targets,\n):\n    crop_loss = crop_ce(crop_logits, crop_targets)\n\n    disease_losses = []\n\n    for crop_idx in range(len(CROPS)):\n        mask = crop_targets == crop_idx\n\n        if mask.any():\n            d_loss = disease_ce(\n                disease_logits[crop_idx][mask],\n                disease_targets[mask],\n            )\n\n            disease_losses.append(d_loss)\n\n    if disease_losses:\n        disease_loss = torch.stack(disease_losses).mean()\n    else:\n        disease_loss = crop_logits.new_tensor(0.0)\n\n    total = (\n        CROP_LOSS_WEIGHT * crop_loss\n        + disease_loss\n    )\n\n    return (\n        total,\n        crop_loss.detach(),\n        disease_loss.detach(),\n    )\n\n\n\n\n# ============================================================\n# 10. METRICS\n# ============================================================\n\ndef classification_metrics(y_true, y_pred):\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n\n    acc = accuracy_score(y_true, y_pred)\n\n    p_macro, r_macro, f_macro, _ = (\n        precision_recall_fscore_support(\n            y_true,\n            y_pred,\n            average="macro",\n            zero_division=0,\n        )\n    )\n\n    p_weight, r_weight, f_weight, _ = (\n        precision_recall_fscore_support(\n            y_true,\n            y_pred,\n            average="weighted",\n            zero_division=0,\n        )\n    )\n\n    return {\n        "accuracy": float(acc),\n        "balanced_accuracy": float(\n            balanced_accuracy_score(y_true, y_pred)\n        ),\n        "macro_precision": float(p_macro),\n        "macro_recall": float(r_macro),\n        "macro_f1": float(f_macro),\n        "weighted_precision": float(p_weight),\n        "weighted_recall": float(r_weight),\n        "weighted_f1": float(f_weight),\n        "mcc": float(matthews_corrcoef(y_true, y_pred)),\n        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),\n    }\n\n\ndef bootstrap_ci(\n    y_true,\n    y_pred,\n    n_classes,\n    repeats=BOOTSTRAP_REPEATS,\n    seed=SEED,\n):\n    """Non-parametric 95% CI over the held-out test samples."""\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n\n    rng = np.random.default_rng(seed)\n    n = len(y_true)\n    labels = np.arange(n_classes)\n\n    acc_values = []\n    macro_f1_values = []\n\n    for _ in range(repeats):\n        idx = rng.integers(0, n, size=n)\n        yt = y_true[idx]\n        yp = y_pred[idx]\n\n        acc_values.append(float(np.mean(yt == yp)))\n        macro_f1_values.append(\n            float(\n                f1_score(\n                    yt,\n                    yp,\n                    labels=labels,\n                    average="macro",\n                    zero_division=0,\n                )\n            )\n        )\n\n    return {\n        "accuracy_ci95_low": float(np.percentile(acc_values, 2.5)),\n        "accuracy_ci95_high": float(np.percentile(acc_values, 97.5)),\n        "macro_f1_ci95_low": float(np.percentile(macro_f1_values, 2.5)),\n        "macro_f1_ci95_high": float(np.percentile(macro_f1_values, 97.5)),\n    }\n\n\ndef count_parameters(model):\n    return sum(\n        p.numel()\n        for p in unwrap_model(model).parameters()\n    )\n\n\ndef save_state_dict(model, path):\n    torch.save(\n        unwrap_model(model).state_dict(),\n        path,\n    )\n\n\ndef checkpoint_size_mb(path):\n    return Path(path).stat().st_size / (1024**2)\n\n\n@torch.no_grad()\ndef benchmark_t4_latency(\n    model,\n    warmup=LATENCY_WARMUP,\n    repeats=LATENCY_REPEATS,\n):\n    """\n    Batch-1 inference benchmark on a single T4.\n    This is a Kaggle-GPU benchmark, NOT Android/mobile latency.\n    """\n    base_model = unwrap_model(model)\n    base_model.eval()\n\n    dummy = torch.randn(\n        1, 3, IMG_SIZE, IMG_SIZE,\n        device=DEVICE,\n    )\n\n    if USE_CHANNELS_LAST:\n        dummy = dummy.contiguous(\n            memory_format=torch.channels_last\n        )\n\n    for _ in range(warmup):\n        _ = base_model(dummy)\n\n    torch.cuda.synchronize()\n\n    elapsed_ms = []\n\n    for _ in range(repeats):\n        start = time.perf_counter()\n        _ = base_model(dummy)\n        torch.cuda.synchronize()\n        elapsed_ms.append(\n            (time.perf_counter() - start) * 1000.0\n        )\n\n    elapsed_ms = np.asarray(elapsed_ms)\n\n    return {\n        "t4_batch1_latency_mean_ms": float(elapsed_ms.mean()),\n        "t4_batch1_latency_p50_ms": float(\n            np.percentile(elapsed_ms, 50)\n        ),\n        "t4_batch1_latency_p95_ms": float(\n            np.percentile(elapsed_ms, 95)\n        ),\n        "t4_batch1_throughput_fps": float(\n            1000.0 / elapsed_ms.mean()\n        ),\n    }\n\n\n\n\n# ============================================================\n# 11. DATALOADERS\n# ============================================================\n\n\n\n_MODEL_DATASET_COUNTS_VERIFIED = False\n\n\ndef verify_model_dataset_counts(train_ds, val_ds, test_ds):\n    """\n    Final minimal safety gate.\n\n    This checks the lengths of the actual Dataset objects passed to the\n    DataLoaders. No SHA-256 hashing, perceptual comparison, source grouping,\n    or physical 73k-file audit is performed.\n    """\n    global _MODEL_DATASET_COUNTS_VERIFIED\n\n    if _MODEL_DATASET_COUNTS_VERIFIED:\n        return\n\n    actual = {\n        "train": len(train_ds),\n        "val": len(val_ds),\n        "test": len(test_ds),\n    }\n\n    expected = EXPECTED_MODEL_SPLIT_COUNTS\n\n    print("\\n" + "=" * 90)\n    print("ACTUAL MODEL DATASET COUNT CHECK")\n    print("=" * 90)\n    print("Train originals :", actual["train"])\n    print("Validation      :", actual["val"])\n    print("Test            :", actual["test"])\n    print("Total originals :", sum(actual.values()))\n\n    if actual != expected:\n        raise RuntimeError(\n            "Actual model Dataset counts do not match the verified final split. "\n            f"Expected {expected}, found {actual}."\n        )\n\n    if sum(actual.values()) != EXPECTED_MODEL_TOTAL_IMAGES:\n        raise RuntimeError(\n            "Actual model Dataset total does not match 18,296."\n        )\n\n    print("MODEL DATASET COUNTS: PASS")\n    print("Duplicate/leakage re-audit: NOT REPEATED")\n    print("=" * 90)\n\n    _MODEL_DATASET_COUNTS_VERIFIED = True\n\n\ndef fast_worker_init_fn(worker_id):\n    """\n    Prevent each DataLoader process from spawning a large PyTorch CPU\n    thread pool. Deterministic PIL/timm online augmentation and preprocessing remain unchanged.\n    """\n    torch.set_num_threads(1)\n\n    worker_seed = (\n        torch.initial_seed()\n        % (\n            2 ** 32\n        )\n    )\n\n    np.random.seed(\n        worker_seed\n    )\n\n    random.seed(\n        worker_seed\n    )\n\n\n\n_WORKERS_AUTOTUNED = False\n\n\ndef _shutdown_loader_iterator(iterator):\n    try:\n        iterator._shutdown_workers()\n    except Exception:\n        pass\n\n\ndef maybe_autotune_num_workers(dataset):\n    """\n    Benchmark a very small number of batches using the actual compressed-file\n    decode + online augmentation pipeline, then keep the fastest worker count\n    for the rest of all eight experiments.\n\n    The benchmark is run only once. RNG is reset afterwards so model training\n    reproducibility is not affected by this startup benchmark.\n    """\n    global NUM_WORKERS\n    global _WORKERS_AUTOTUNED\n\n    if _WORKERS_AUTOTUNED or not AUTO_TUNE_NUM_WORKERS:\n        return\n\n    print("\\n" + "=" * 90)\n    print("ONE-TIME DATALOADER WORKER AUTOTUNE")\n    print("=" * 90)\n    print("Candidates:", WORKER_TUNE_CANDIDATES)\n    print("Prefetch factor:", PREFETCH_FACTOR)\n\n    results = []\n\n    for workers in WORKER_TUNE_CANDIDATES:\n        kwargs = dict(\n            batch_size=BATCH_SIZE,\n            shuffle=False,\n            num_workers=workers,\n            pin_memory=True,\n            persistent_workers=False,\n            worker_init_fn=fast_worker_init_fn,\n        )\n\n        if workers > 0:\n            kwargs["prefetch_factor"] = PREFETCH_FACTOR\n\n        tune_loader = DataLoader(\n            dataset,\n            **kwargs,\n        )\n\n        iterator = iter(tune_loader)\n\n        try:\n            for _ in range(WORKER_TUNE_WARMUP_BATCHES):\n                next(iterator)\n\n            start = time.perf_counter()\n            completed = 0\n\n            for _ in range(WORKER_TUNE_TIMED_BATCHES):\n                next(iterator)\n                completed += 1\n\n            elapsed = time.perf_counter() - start\n\n            images_seen = (\n                completed\n                * BATCH_SIZE\n            )\n\n            throughput = (\n                images_seen\n                / max(elapsed, 1e-9)\n            )\n\n            results.append(\n                (throughput, workers)\n            )\n\n            print(\n                f"workers={workers}: "\n                f"{throughput:.1f} images/s "\n                f"({elapsed:.2f}s for {completed} timed batches)"\n            )\n\n        except StopIteration:\n            pass\n        finally:\n            _shutdown_loader_iterator(iterator)\n            del iterator\n            del tune_loader\n            gc.collect()\n\n    if not results:\n        raise RuntimeError(\n            "Could not benchmark any DataLoader worker candidate."\n        )\n\n    results.sort(\n        reverse=True\n    )\n\n    best_throughput, best_workers = results[0]\n    NUM_WORKERS = int(best_workers)\n    _WORKERS_AUTOTUNED = True\n\n    # The tuning loaders consume RNG states for worker seeding / augmentation.\n    # Reset before the real experiment loader and model are created.\n    seed_everything(SEED)\n\n    print(\n        f"Selected NUM_WORKERS={NUM_WORKERS} "\n        f"({best_throughput:.1f} images/s CPU pipeline throughput)"\n    )\n    print("=" * 90)\n\n\ndef loader_kwargs():\n    kwargs = dict(\n        num_workers=NUM_WORKERS,\n        pin_memory=True,\n        persistent_workers=NUM_WORKERS > 0,\n        worker_init_fn=fast_worker_init_fn,\n    )\n\n    if NUM_WORKERS > 0:\n        kwargs["prefetch_factor"] = PREFETCH_FACTOR\n\n    return kwargs\n\n\ndef build_flat_loaders(backbone_key):\n    train_tf, eval_tf = make_transforms(backbone_key)\n\n    train_h = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "train",\n        train_tf,\n    )\n    val_h = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "val",\n        eval_tf,\n    )\n    test_h = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "test",\n        eval_tf,\n    )\n\n    train_ds = FlatLeafDataset(train_h)\n    val_ds = FlatLeafDataset(val_h)\n    test_ds = FlatLeafDataset(test_h)\n\n    verify_model_dataset_counts(\n        train_ds,\n        val_ds,\n        test_ds,\n    )\n\n    maybe_autotune_num_workers(\n        train_ds\n    )\n\n    sampler = build_flat_balanced_sampler(train_ds)\n\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=BATCH_SIZE,\n        sampler=sampler,\n        **loader_kwargs(),\n    )\n\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=EVAL_BATCH_SIZE,\n        shuffle=False,\n        **loader_kwargs(),\n    )\n\n    test_loader = DataLoader(\n        test_ds,\n        batch_size=EVAL_BATCH_SIZE,\n        shuffle=False,\n        **loader_kwargs(),\n    )\n\n    return (\n        train_loader,\n        val_loader,\n        test_loader,\n        train_ds.global_classes,\n    )\n\n\ndef build_hier_loaders(backbone_key):\n    train_tf, eval_tf = make_transforms(backbone_key)\n\n    train_ds = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "train",\n        train_tf,\n    )\n    val_ds = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "val",\n        eval_tf,\n    )\n    test_ds = HierarchicalLeafDataset(\n        DATA_ROOT,\n        "test",\n        eval_tf,\n    )\n\n    verify_model_dataset_counts(\n        train_ds,\n        val_ds,\n        test_ds,\n    )\n\n    maybe_autotune_num_workers(\n        train_ds\n    )\n\n    sampler = build_hierarchical_balanced_sampler(train_ds)\n\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=BATCH_SIZE,\n        sampler=sampler,\n        **loader_kwargs(),\n    )\n\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=EVAL_BATCH_SIZE,\n        shuffle=False,\n        **loader_kwargs(),\n    )\n\n    test_loader = DataLoader(\n        test_ds,\n        batch_size=EVAL_BATCH_SIZE,\n        shuffle=False,\n        **loader_kwargs(),\n    )\n\n    crop_class_counts = [\n        len(train_ds.crop_classes[crop])\n        for crop in CROPS\n    ]\n\n    return (\n        train_loader,\n        val_loader,\n        test_loader,\n        train_ds,\n        val_ds,\n        test_ds,\n        crop_class_counts,\n    )\n\n\n\n\n# ============================================================\n# 12. FLAT TRAINING — SPEED OPTIMIZED, SAME METRICS\n# ============================================================\n\ndef _to_gpu_images(images):\n    """\n    Transfer images to the single visible T4.\n\n    Fast-cache path:\n      pinned CPU uint8 -> async H2D -> float/255 -> model normalization on GPU.\n\n    Normal path:\n      the existing CPU transform already returns normalized float tensors.\n    """\n    global _GPU_NORM_MEAN\n    global _GPU_NORM_STD\n\n    if USE_MULTI_GPU and USE_DIRECT_DP_CPU_SCATTER:\n        return images\n\n    images = images.to(\n        DEVICE,\n        non_blocking=True,\n    )\n\n    if images.dtype == torch.uint8:\n        if ACTIVE_INPUT_MEAN is None or ACTIVE_INPUT_STD is None:\n            raise RuntimeError(\n                "GPU normalization requested before model mean/std were resolved."\n            )\n\n        images = images.to(\n            dtype=torch.float32\n        ).mul_(\n            1.0 / 255.0\n        )\n\n        if _GPU_NORM_MEAN is None:\n            _GPU_NORM_MEAN = torch.tensor(\n                ACTIVE_INPUT_MEAN,\n                dtype=torch.float32,\n                device=DEVICE,\n            ).view(\n                1, 3, 1, 1\n            )\n\n            _GPU_NORM_STD = torch.tensor(\n                ACTIVE_INPUT_STD,\n                dtype=torch.float32,\n                device=DEVICE,\n            ).view(\n                1, 3, 1, 1\n            )\n\n        images.sub_(\n            _GPU_NORM_MEAN\n        ).div_(\n            _GPU_NORM_STD\n        )\n\n    return images\n\n\ndef _update_gpu_confusion(cm, y_true, y_pred):\n    n = cm.shape[0]\n    encoded = (\n        y_true.to(torch.int64) * n\n        + y_pred.to(torch.int64)\n    )\n    cm += torch.bincount(\n        encoded,\n        minlength=n * n,\n    ).reshape(n, n)\n\n\ndef _basic_metrics_from_gpu_confusion(cm):\n    cmf = cm.to(torch.float32)\n    tp = torch.diag(cmf)\n    total = cmf.sum()\n\n    accuracy = torch.where(\n        total > 0,\n        tp.sum() / total,\n        torch.zeros((), device=cm.device),\n    )\n\n    support = cmf.sum(dim=1)\n    predicted = cmf.sum(dim=0)\n\n    precision = torch.where(\n        predicted > 0,\n        tp / predicted.clamp_min(1.0),\n        torch.zeros_like(tp),\n    )\n\n    recall = torch.where(\n        support > 0,\n        tp / support.clamp_min(1.0),\n        torch.zeros_like(tp),\n    )\n\n    denom = precision + recall\n    f1 = torch.where(\n        denom > 0,\n        2.0 * precision * recall / denom,\n        torch.zeros_like(denom),\n    )\n\n    return {\n        "accuracy": float(accuracy.item()),\n        "macro_f1": float(f1.mean().item()),\n    }\n\ndef run_flat_epoch(\n    model,\n    loader,\n    optimizer=None,\n    scaler=None,\n):\n    train_mode = optimizer is not None\n    model.train(train_mode)\n\n    total_loss = torch.zeros(\n        (),\n        device=DEVICE,\n        dtype=torch.float32,\n    )\n\n    n_classes = len(loader.dataset.global_classes)\n\n    cm = torch.zeros(\n        (n_classes, n_classes),\n        device=DEVICE,\n        dtype=torch.int64,\n    )\n\n    grad_context = (\n        torch.enable_grad\n        if train_mode\n        else torch.inference_mode\n    )\n\n    with grad_context():\n        for images, targets in tqdm(\n            loader,\n            leave=False,\n            mininterval=TQDM_MININTERVAL,\n        ):\n            images = _to_gpu_images(images)\n\n            targets = targets.to(\n                DEVICE,\n                non_blocking=True,\n            )\n\n            if train_mode:\n                optimizer.zero_grad(\n                    set_to_none=True\n                )\n\n            with torch.cuda.amp.autocast(\n                enabled=USE_AMP\n            ):\n                logits = model(images)\n                loss = flat_ce(\n                    logits,\n                    targets,\n                )\n\n            if train_mode:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n\n            batch_size = targets.size(0)\n\n            total_loss += (\n                loss.detach().float()\n                * batch_size\n            )\n\n            pred = logits.detach().argmax(\n                dim=1\n            )\n\n            _update_gpu_confusion(\n                cm,\n                targets,\n                pred,\n            )\n\n    metrics = _basic_metrics_from_gpu_confusion(\n        cm\n    )\n\n    metrics["loss"] = float(\n        (\n            total_loss\n            / len(loader.dataset)\n        ).item()\n    )\n\n    return metrics\n\n\n@torch.inference_mode()\ndef evaluate_flat_test(\n    model,\n    loader,\n    class_names,\n):\n    model.eval()\n\n    y_true_gpu = []\n    y_pred_gpu = []\n    confidence_gpu = []\n\n    for images, targets in tqdm(\n        loader,\n        leave=False,\n        mininterval=TQDM_MININTERVAL,\n    ):\n        images = _to_gpu_images(\n            images\n        )\n\n        logits = model(\n            images\n        )\n\n        probs = torch.softmax(\n            logits,\n            dim=1,\n        )\n\n        conf, pred = probs.max(\n            dim=1\n        )\n\n        y_true_gpu.append(\n            targets.to(\n                DEVICE,\n                non_blocking=True,\n            )\n        )\n\n        y_pred_gpu.append(\n            pred.detach()\n        )\n\n        confidence_gpu.append(\n            conf.detach()\n        )\n\n    y_true = (\n        torch.cat(\n            y_true_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    y_pred = (\n        torch.cat(\n            y_pred_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    confidences = (\n        torch.cat(\n            confidence_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    metrics = classification_metrics(\n        y_true,\n        y_pred,\n    )\n\n    report = classification_report(\n        y_true,\n        y_pred,\n        target_names=class_names,\n        output_dict=True,\n        zero_division=0,\n    )\n\n    cm = confusion_matrix(\n        y_true,\n        y_pred,\n    )\n\n    return (\n        metrics,\n        report,\n        cm,\n        y_true,\n        y_pred,\n        confidences,\n    )\n\n\n# ============================================================\n# 13. HIERARCHICAL EPOCH METRICS — VECTORIZED FAST VERSION\n# ============================================================\n\ndef _crop_global_offsets(dataset):\n    offsets = []\n    running = 0\n\n    for crop in CROPS:\n        offsets.append(\n            running\n        )\n\n        running += len(\n            dataset.crop_classes[\n                crop\n            ]\n        )\n\n    return torch.tensor(\n        offsets,\n        dtype=torch.long,\n        device=DEVICE,\n    )\n\n\ndef _vectorized_routed_prediction(\n    crop_logits,\n    disease_logits,\n    dataset,\n):\n    """\n    TRUE sequential routing:\n      predicted crop -> corresponding disease head -> global class index.\n\n    Same mathematical prediction as the previous per-sample loop,\n    implemented batch-wise to avoid CUDA synchronization.\n    """\n    crop_prediction = crop_logits.argmax(\n        dim=1\n    )\n\n    global_prediction = torch.empty(\n        crop_prediction.shape[0],\n        dtype=torch.long,\n        device=crop_prediction.device,\n    )\n\n    offsets = _crop_global_offsets(\n        dataset\n    )\n\n    for crop_idx in range(\n        len(CROPS)\n    ):\n        mask = (\n            crop_prediction\n            == crop_idx\n        )\n\n        # Do not call .item(); indexing an empty mask is valid.\n        local_prediction = (\n            disease_logits[\n                crop_idx\n            ][mask]\n            .argmax(\n                dim=1\n            )\n        )\n\n        global_prediction[\n            mask\n        ] = (\n            offsets[\n                crop_idx\n            ]\n            + local_prediction\n        )\n\n    return (\n        crop_prediction,\n        global_prediction,\n    )\n\n\ndef run_hier_epoch(\n    model,\n    loader,\n    optimizer=None,\n    scaler=None,\n):\n    train_mode = optimizer is not None\n    model.train(train_mode)\n\n    total_loss = torch.zeros(\n        (),\n        device=DEVICE,\n        dtype=torch.float32,\n    )\n    total_crop_loss = torch.zeros(\n        (),\n        device=DEVICE,\n        dtype=torch.float32,\n    )\n    total_disease_loss = torch.zeros(\n        (),\n        device=DEVICE,\n        dtype=torch.float32,\n    )\n\n    ds = loader.dataset\n    if hasattr(ds, "base"):\n        ds = ds.base\n\n    n_global = len(ds.global_classes)\n\n    crop_cm = torch.zeros(\n        (len(CROPS), len(CROPS)),\n        device=DEVICE,\n        dtype=torch.int64,\n    )\n\n    end_cm = torch.zeros(\n        (n_global, n_global),\n        device=DEVICE,\n        dtype=torch.int64,\n    )\n\n    oracle_cms = {\n        crop_idx: torch.zeros(\n            (\n                len(ds.crop_classes[crop]),\n                len(ds.crop_classes[crop]),\n            ),\n            device=DEVICE,\n            dtype=torch.int64,\n        )\n        for crop_idx, crop in enumerate(CROPS)\n    }\n\n    grad_context = (\n        torch.enable_grad\n        if train_mode\n        else torch.inference_mode\n    )\n\n    with grad_context():\n        for (\n            images,\n            crop_t,\n            disease_t,\n            global_t,\n        ) in tqdm(\n            loader,\n            leave=False,\n            mininterval=TQDM_MININTERVAL,\n        ):\n            images = _to_gpu_images(images)\n\n            crop_t_gpu = crop_t.to(\n                DEVICE,\n                non_blocking=True,\n            )\n            disease_t_gpu = disease_t.to(\n                DEVICE,\n                non_blocking=True,\n            )\n            global_t_gpu = global_t.to(\n                DEVICE,\n                non_blocking=True,\n            )\n\n            if train_mode:\n                optimizer.zero_grad(\n                    set_to_none=True\n                )\n\n            with torch.cuda.amp.autocast(\n                enabled=USE_AMP\n            ):\n                (\n                    crop_logits,\n                    disease_logits,\n                ) = model(images)\n\n                (\n                    loss,\n                    crop_loss,\n                    disease_loss,\n                ) = hierarchical_loss(\n                    crop_logits,\n                    disease_logits,\n                    crop_t_gpu,\n                    disease_t_gpu,\n                )\n\n            if train_mode:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n\n            batch_size = crop_t_gpu.size(0)\n\n            total_loss += (\n                loss.detach().float()\n                * batch_size\n            )\n            total_crop_loss += (\n                crop_loss.detach().float()\n                * batch_size\n            )\n            total_disease_loss += (\n                disease_loss.detach().float()\n                * batch_size\n            )\n\n            (\n                crop_prediction,\n                global_prediction,\n            ) = _vectorized_routed_prediction(\n                crop_logits.detach(),\n                [\n                    output.detach()\n                    for output in disease_logits\n                ],\n                ds,\n            )\n\n            _update_gpu_confusion(\n                crop_cm,\n                crop_t_gpu,\n                crop_prediction,\n            )\n\n            _update_gpu_confusion(\n                end_cm,\n                global_t_gpu,\n                global_prediction,\n            )\n\n            # Oracle Stage-2 metrics use the TRUE crop.\n            for crop_idx in range(len(CROPS)):\n                mask = (\n                    crop_t_gpu\n                    == crop_idx\n                )\n\n                if mask.any():\n                    oracle_pred = (\n                        disease_logits[crop_idx][mask]\n                        .detach()\n                        .argmax(dim=1)\n                    )\n\n                    _update_gpu_confusion(\n                        oracle_cms[crop_idx],\n                        disease_t_gpu[mask],\n                        oracle_pred,\n                    )\n\n    crop_metrics = _basic_metrics_from_gpu_confusion(\n        crop_cm\n    )\n\n    end_metrics = _basic_metrics_from_gpu_confusion(\n        end_cm\n    )\n\n    per_crop = {\n        crop: _basic_metrics_from_gpu_confusion(\n            oracle_cms[crop_idx]\n        )\n        for crop_idx, crop in enumerate(CROPS)\n    }\n\n    dataset_size = len(loader.dataset)\n\n    return {\n        "loss": float(\n            (\n                total_loss\n                / dataset_size\n            ).item()\n        ),\n        "crop_loss": float(\n            (\n                total_crop_loss\n                / dataset_size\n            ).item()\n        ),\n        "disease_loss": float(\n            (\n                total_disease_loss\n                / dataset_size\n            ).item()\n        ),\n\n        "crop_accuracy": crop_metrics["accuracy"],\n        "crop_macro_f1": crop_metrics["macro_f1"],\n\n        "end_to_end_accuracy": end_metrics["accuracy"],\n        "end_to_end_macro_f1": end_metrics["macro_f1"],\n\n        "potato_accuracy": per_crop["potato"]["accuracy"],\n        "potato_macro_f1": per_crop["potato"]["macro_f1"],\n\n        "rice_accuracy": per_crop["rice"]["accuracy"],\n        "rice_macro_f1": per_crop["rice"]["macro_f1"],\n\n        "tomato_accuracy": per_crop["tomato"]["accuracy"],\n        "tomato_macro_f1": per_crop["tomato"]["macro_f1"],\n    }\n\n\n# ============================================================\n# 14. FULL HIERARCHICAL TEST EVALUATION — VECTORIZED FAST VERSION\n# ============================================================\n\n@torch.inference_mode()\ndef evaluate_hierarchical_test(\n    model,\n    loader,\n    dataset,\n):\n    model.eval()\n\n    crop_true_gpu = []\n    crop_pred_gpu = []\n    crop_confidence_gpu = []\n\n    oracle_true_gpu = {\n        crop_idx: []\n        for crop_idx in range(\n            len(CROPS)\n        )\n    }\n\n    oracle_pred_gpu = {\n        crop_idx: []\n        for crop_idx in range(\n            len(CROPS)\n        )\n    }\n\n    oracle_confidence_gpu = {\n        crop_idx: []\n        for crop_idx in range(\n            len(CROPS)\n        )\n    }\n\n    global_true_gpu = []\n    global_pred_gpu = []\n    end_confidence_gpu = []\n\n    offsets = _crop_global_offsets(\n        dataset\n    )\n\n    for (\n        images,\n        crop_t,\n        disease_t,\n        global_t,\n    ) in tqdm(\n        loader,\n        leave=False,\n        mininterval=TQDM_MININTERVAL,\n    ):\n        images = _to_gpu_images(\n            images\n        )\n\n        crop_t_gpu = crop_t.to(\n            DEVICE,\n            non_blocking=True,\n        )\n\n        disease_t_gpu = disease_t.to(\n            DEVICE,\n            non_blocking=True,\n        )\n\n        global_t_gpu = global_t.to(\n            DEVICE,\n            non_blocking=True,\n        )\n\n        with torch.cuda.amp.autocast(\n            enabled=USE_AMP\n        ):\n            (\n                crop_logits,\n                disease_logits,\n            ) = model(\n                images\n            )\n\n        crop_prob = torch.softmax(\n            crop_logits,\n            dim=1,\n        )\n\n        (\n            crop_confidence,\n            crop_prediction,\n        ) = crop_prob.max(\n            dim=1\n        )\n\n        global_prediction = torch.empty(\n            crop_prediction.shape[0],\n            dtype=torch.long,\n            device=DEVICE,\n        )\n\n        sequential_confidence = torch.empty(\n            crop_prediction.shape[0],\n            dtype=crop_prob.dtype,\n            device=DEVICE,\n        )\n\n        for crop_idx in range(\n            len(CROPS)\n        ):\n            predicted_mask = (\n                crop_prediction\n                == crop_idx\n            )\n\n            routed_prob = torch.softmax(\n                disease_logits[\n                    crop_idx\n                ][predicted_mask],\n                dim=1,\n            )\n\n            (\n                routed_confidence,\n                routed_prediction,\n            ) = routed_prob.max(\n                dim=1\n            )\n\n            global_prediction[\n                predicted_mask\n            ] = (\n                offsets[\n                    crop_idx\n                ]\n                + routed_prediction\n            )\n\n            sequential_confidence[\n                predicted_mask\n            ] = (\n                crop_confidence[\n                    predicted_mask\n                ]\n                * routed_confidence\n            )\n\n            # Oracle disease evaluation uses TRUE crop.\n            true_mask = (\n                crop_t_gpu\n                == crop_idx\n            )\n\n            oracle_prob = torch.softmax(\n                disease_logits[\n                    crop_idx\n                ][true_mask],\n                dim=1,\n            )\n\n            (\n                oracle_confidence,\n                oracle_prediction,\n            ) = oracle_prob.max(\n                dim=1\n            )\n\n            oracle_true_gpu[\n                crop_idx\n            ].append(\n                disease_t_gpu[\n                    true_mask\n                ].detach()\n            )\n\n            oracle_pred_gpu[\n                crop_idx\n            ].append(\n                oracle_prediction.detach()\n            )\n\n            oracle_confidence_gpu[\n                crop_idx\n            ].append(\n                oracle_confidence.detach()\n            )\n\n        crop_true_gpu.append(\n            crop_t_gpu.detach()\n        )\n\n        crop_pred_gpu.append(\n            crop_prediction.detach()\n        )\n\n        crop_confidence_gpu.append(\n            crop_confidence.detach()\n        )\n\n        global_true_gpu.append(\n            global_t_gpu.detach()\n        )\n\n        global_pred_gpu.append(\n            global_prediction.detach()\n        )\n\n        end_confidence_gpu.append(\n            sequential_confidence.detach()\n        )\n\n    crop_true = (\n        torch.cat(\n            crop_true_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    crop_pred = (\n        torch.cat(\n            crop_pred_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    crop_confidences = (\n        torch.cat(\n            crop_confidence_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    global_true = (\n        torch.cat(\n            global_true_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    global_pred = (\n        torch.cat(\n            global_pred_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    end_confidences = (\n        torch.cat(\n            end_confidence_gpu\n        )\n        .cpu()\n        .numpy()\n        .tolist()\n    )\n\n    oracle_true = {}\n    oracle_pred = {}\n    oracle_confidence = {}\n\n    for crop_idx in range(\n        len(CROPS)\n    ):\n        true_parts = [\n            tensor\n            for tensor\n            in oracle_true_gpu[\n                crop_idx\n            ]\n            if tensor.numel() > 0\n        ]\n\n        pred_parts = [\n            tensor\n            for tensor\n            in oracle_pred_gpu[\n                crop_idx\n            ]\n            if tensor.numel() > 0\n        ]\n\n        conf_parts = [\n            tensor\n            for tensor\n            in oracle_confidence_gpu[\n                crop_idx\n            ]\n            if tensor.numel() > 0\n        ]\n\n        oracle_true[\n            crop_idx\n        ] = (\n            torch.cat(\n                true_parts\n            )\n            .cpu()\n            .numpy()\n            .tolist()\n        )\n\n        oracle_pred[\n            crop_idx\n        ] = (\n            torch.cat(\n                pred_parts\n            )\n            .cpu()\n            .numpy()\n            .tolist()\n        )\n\n        oracle_confidence[\n            crop_idx\n        ] = (\n            torch.cat(\n                conf_parts\n            )\n            .cpu()\n            .numpy()\n            .tolist()\n        )\n\n    crop_metrics = classification_metrics(\n        crop_true,\n        crop_pred,\n    )\n\n    end_metrics = classification_metrics(\n        global_true,\n        global_pred,\n    )\n\n    per_crop = {}\n\n    for crop_idx, crop in enumerate(\n        CROPS\n    ):\n        per_crop[\n            crop\n        ] = classification_metrics(\n            oracle_true[\n                crop_idx\n            ],\n            oracle_pred[\n                crop_idx\n            ],\n        )\n\n    crop_cm = confusion_matrix(\n        crop_true,\n        crop_pred,\n    )\n\n    global_cm = confusion_matrix(\n        global_true,\n        global_pred,\n    )\n\n    per_crop_cm = {}\n\n    for crop_idx, crop in enumerate(\n        CROPS\n    ):\n        per_crop_cm[\n            crop\n        ] = confusion_matrix(\n            oracle_true[\n                crop_idx\n            ],\n            oracle_pred[\n                crop_idx\n            ],\n        )\n\n    crop_report = classification_report(\n        crop_true,\n        crop_pred,\n        target_names=[\n            crop.title()\n            for crop in CROPS\n        ],\n        output_dict=True,\n        zero_division=0,\n    )\n\n    per_crop_report = {}\n\n    for crop_idx, crop in enumerate(\n        CROPS\n    ):\n        per_crop_report[\n            crop\n        ] = classification_report(\n            oracle_true[\n                crop_idx\n            ],\n            oracle_pred[\n                crop_idx\n            ],\n            target_names=dataset.crop_classes[\n                crop\n            ],\n            output_dict=True,\n            zero_division=0,\n        )\n\n    global_report = classification_report(\n        global_true,\n        global_pred,\n        target_names=dataset.global_classes,\n        output_dict=True,\n        zero_division=0,\n    )\n\n    return {\n        "crop": crop_metrics,\n        "oracle_per_crop": per_crop,\n        "end_to_end": end_metrics,\n\n        "crop_cm": crop_cm,\n        "per_crop_cm": per_crop_cm,\n        "global_cm": global_cm,\n\n        "crop_report": crop_report,\n        "per_crop_report": per_crop_report,\n        "global_report": global_report,\n\n        "crop_true": crop_true,\n        "crop_pred": crop_pred,\n\n        "global_true": global_true,\n        "global_pred": global_pred,\n\n        "crop_confidences": crop_confidences,\n        "oracle_confidence": oracle_confidence,\n        "end_confidences": end_confidences,\n    }\n\n\n# ============================================================\n# 15. PLOTTING UTILITIES — ALL 300-DPI PNG\n# ============================================================\n\ndef save_line_plot(\n    x,\n    series,\n    title,\n    xlabel,\n    ylabel,\n    path,\n):\n    plt.figure(figsize=(8, 5))\n\n    for label, values in series.items():\n        plt.plot(\n            x,\n            values,\n            label=label,\n        )\n\n    plt.xlabel(xlabel)\n    plt.ylabel(ylabel)\n    plt.title(title)\n\n    if len(series) > 1:\n        plt.legend()\n\n    plt.grid(alpha=0.25)\n    plt.tight_layout()\n\n    plt.savefig(\n        path,\n        dpi=300,\n        bbox_inches="tight",\n    )\n\n    plt.close()\n\n\ndef plot_confusion_matrix(\n    cm,\n    labels,\n    title,\n    path,\n    normalize=False,\n):\n    cm = np.array(cm)\n\n    if normalize:\n        cm = cm.astype(float)\n\n        row_sums = cm.sum(\n            axis=1,\n            keepdims=True,\n        )\n\n        cm = np.divide(\n            cm,\n            row_sums,\n            out=np.zeros_like(cm),\n            where=row_sums != 0,\n        )\n\n    plt.figure(\n        figsize=(\n            max(7, len(labels) * 0.75),\n            max(6, len(labels) * 0.65),\n        )\n    )\n\n    plt.imshow(\n        cm,\n        interpolation="nearest",\n        cmap="Blues",\n    )\n\n    plt.title(title)\n    plt.colorbar()\n\n    tick_marks = np.arange(len(labels))\n\n    plt.xticks(\n        tick_marks,\n        labels,\n        rotation=90,\n        fontsize=8,\n    )\n\n    plt.yticks(\n        tick_marks,\n        labels,\n        fontsize=8,\n    )\n\n    threshold = (\n        cm.max() / 2.0\n        if cm.size and cm.max() > 0\n        else 0\n    )\n\n    for i in range(cm.shape[0]):\n        for j in range(cm.shape[1]):\n            if normalize:\n                text = f"{cm[i, j]:.2f}"\n            else:\n                text = f"{int(cm[i, j])}"\n\n            plt.text(\n                j,\n                i,\n                text,\n                horizontalalignment="center",\n                verticalalignment="center",\n                fontsize=7,\n                color=(\n                    "white"\n                    if cm[i, j] > threshold\n                    else "black"\n                ),\n            )\n\n    plt.xlabel("Predicted class")\n    plt.ylabel("True class")\n    plt.tight_layout()\n\n    plt.savefig(\n        path,\n        dpi=300,\n        bbox_inches="tight",\n    )\n\n    plt.close()\n\n\ndef report_to_dataframe(\n    report,\n    class_names,\n):\n    rows = []\n\n    for cls in class_names:\n        r = report[cls]\n\n        rows.append(\n            {\n                "class": cls,\n                "precision": r["precision"],\n                "recall": r["recall"],\n                "f1-score": r["f1-score"],\n                "support": r["support"],\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef plot_per_class_metrics(\n    report,\n    class_names,\n    title,\n    path,\n):\n    df = report_to_dataframe(\n        report,\n        class_names,\n    )\n\n    y = np.arange(len(df))\n    height = 0.24\n\n    plt.figure(\n        figsize=(\n            11,\n            max(6, len(df) * 0.45),\n        )\n    )\n\n    plt.barh(\n        y - height,\n        df["precision"],\n        height=height,\n        label="Precision",\n    )\n\n    plt.barh(\n        y,\n        df["recall"],\n        height=height,\n        label="Recall",\n    )\n\n    plt.barh(\n        y + height,\n        df["f1-score"],\n        height=height,\n        label="F1-score",\n    )\n\n    plt.yticks(y, df["class"], fontsize=8)\n    plt.xlabel("Score")\n    plt.xlim(0, 1.0)\n    plt.title(title)\n    plt.legend()\n    plt.tight_layout()\n\n    plt.savefig(\n        path,\n        dpi=300,\n        bbox_inches="tight",\n    )\n\n    plt.close()\n\n    return df\n\n\ndef plot_confidence_distribution(\n    y_true,\n    y_pred,\n    confidence,\n    title,\n    path,\n):\n    y_true = np.array(y_true)\n    y_pred = np.array(y_pred)\n    confidence = np.array(confidence)\n\n    correct = confidence[y_true == y_pred]\n    incorrect = confidence[y_true != y_pred]\n\n    plt.figure(figsize=(8, 5))\n\n    if len(correct) > 0:\n        plt.hist(\n            correct,\n            bins=20,\n            alpha=0.65,\n            label=f"Correct (n={len(correct)})",\n        )\n\n    if len(incorrect) > 0:\n        plt.hist(\n            incorrect,\n            bins=20,\n            alpha=0.65,\n            label=f"Incorrect (n={len(incorrect)})",\n        )\n\n    plt.xlabel("Prediction confidence")\n    plt.ylabel("Number of samples")\n    plt.title(title)\n    plt.legend()\n    plt.tight_layout()\n\n    plt.savefig(\n        path,\n        dpi=300,\n        bbox_inches="tight",\n    )\n\n    plt.close()\n\n\n\n\n# ============================================================\n# 16. FLAT EXPERIMENT\n# ============================================================\n\n\ndef save_history_csv(history, path):\n    rows = []\n    for item in history:\n        row = {\n            "epoch": item.get("epoch"),\n            "lr": item.get("lr"),\n        }\n        for split in ["train", "val"]:\n            metrics = item.get(split, {}) or {}\n            for key, value in metrics.items():\n                if isinstance(value, (int, float, np.integer, np.floating)):\n                    row[f"{split}_{key}"] = float(value)\n        rows.append(row)\n    pd.DataFrame(rows).to_csv(path, index=False)\n\n\ndef write_experiment_config(exp_name, family, backbone_key, variant):\n    payload = {\n        "experiment": exp_name,\n        "family": family,\n        "backbone_key": backbone_key,\n        "resolved_backbone": RESOLVED_BACKBONES.get(backbone_key),\n        "variant": variant,\n        "seed": SEED,\n        "image_size": IMG_SIZE,\n        "train_batch_size": BATCH_SIZE,\n        "eval_batch_size": EVAL_BATCH_SIZE,\n        "max_epochs": MAX_EPOCHS,\n        "patience": PATIENCE,\n        "min_delta": MIN_DELTA,\n        "optimizer": "AdamW",\n        "learning_rate": LR,\n        "weight_decay": WEIGHT_DECAY,\n        "crop_loss_weight": CROP_LOSS_WEIGHT,\n        "crop_label_smoothing": CROP_LABEL_SMOOTHING,\n        "flat_label_smoothing": FLAT_LABEL_SMOOTHING,\n        "online_augmentation": False,\n        "offline_augmented_files_used": False,\n        "weighted_balanced_sampling": True,\n        "amp": USE_AMP,\n        "channels_last": USE_CHANNELS_LAST,\n        "dataset_counts": EXPECTED_MODEL_SPLIT_COUNTS,\n        "dataset_total": EXPECTED_MODEL_TOTAL_IMAGES,\n    }\n    (OUTPUT_ROOT / "experiment_config.json").write_text(\n        json.dumps(payload, indent=2), encoding="utf-8"\n    )\n\n\ndef train_flat_experiment(\n    exp_name,\n    backbone_key,\n):\n    print("\\n" + "=" * 90)\n    print("EXPERIMENT:", exp_name)\n    print("TYPE: Flat 16-class")\n    print("BACKBONE:", backbone_key)\n    print("=" * 90)\n\n    seed_everything(SEED)\n\n    exp_dir = OUTPUT_ROOT / exp_name\n    exp_dir.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n\n    (\n        train_loader,\n        val_loader,\n        test_loader,\n        class_names,\n    ) = build_flat_loaders(backbone_key)\n\n    model = FlatClassifier(\n        backbone_key,\n        n_classes=len(class_names),\n    )\n\n    model = maybe_parallel(model)\n\n    optimizer = make_fast_adamw(\n        model\n    )\n\n    scheduler = (\n        torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer,\n            T_max=MAX_EPOCHS,\n        )\n    )\n\n    scaler = torch.cuda.amp.GradScaler(\n        enabled=USE_AMP\n    )\n\n    best_val = float("inf")\n    best_epoch = 0\n    patience_left = PATIENCE\n\n    history = []\n\n    ckpt = (\n        CHECKPOINT_ROOT\n        / f"{exp_name}_best.pth"\n    )\n\n    start = time.time()\n\n    for epoch in range(1, MAX_EPOCHS + 1):\n        train_metrics = run_flat_epoch(\n            model,\n            train_loader,\n            optimizer,\n            scaler,\n        )\n\n        val_metrics = run_flat_epoch(\n            model,\n            val_loader,\n        )\n\n        scheduler.step()\n\n        history.append(\n            {\n                "epoch": epoch,\n                "lr": optimizer.param_groups[0]["lr"],\n                "train": train_metrics,\n                "val": val_metrics,\n            }\n        )\n\n        print(\n            f"[{epoch:03d}/{MAX_EPOCHS}] "\n            f"train_loss={train_metrics[\'loss\']:.4f} "\n            f"train_acc={train_metrics[\'accuracy\']:.4f} | "\n            f"val_loss={val_metrics[\'loss\']:.4f} "\n            f"val_acc={val_metrics[\'accuracy\']:.4f} "\n            f"val_macroF1={val_metrics[\'macro_f1\']:.4f}"\n        )\n\n        if (\n            best_val - val_metrics["loss"]\n            > MIN_DELTA\n        ):\n            best_val = val_metrics["loss"]\n            best_epoch = epoch\n            patience_left = PATIENCE\n\n            save_state_dict(\n                model,\n                ckpt,\n            )\n\n        else:\n            patience_left -= 1\n\n        if patience_left <= 0:\n            print(\n                f"Early stopping at epoch {epoch}."\n            )\n            break\n\n    train_seconds = time.time() - start\n\n    unwrap_model(model).load_state_dict(\n        torch.load(\n            ckpt,\n            map_location=DEVICE,\n        )\n    )\n\n    (\n        test_metrics,\n        report,\n        cm,\n        y_true,\n        y_pred,\n        confidence,\n    ) = evaluate_flat_test(\n        model,\n        test_loader,\n        class_names,\n    )\n\n    ci = bootstrap_ci(\n        y_true,\n        y_pred,\n        n_classes=len(class_names),\n    )\n\n    latency = benchmark_t4_latency(model)\n\n    result = {\n        "experiment": exp_name,\n        "family": "flat",\n        "backbone": backbone_key,\n        "variant": "flat_16class",\n\n        "best_epoch": best_epoch,\n        "training_seconds": train_seconds,\n\n        "parameters": count_parameters(model),\n        "checkpoint_mb": checkpoint_size_mb(ckpt),\n\n        "test_accuracy": test_metrics["accuracy"],\n        "test_balanced_accuracy": test_metrics["balanced_accuracy"],\n        "test_macro_precision": test_metrics["macro_precision"],\n        "test_macro_recall": test_metrics["macro_recall"],\n        "test_macro_f1": test_metrics["macro_f1"],\n        "test_weighted_f1": test_metrics["weighted_f1"],\n        "test_mcc": test_metrics["mcc"],\n        "test_cohen_kappa": test_metrics["cohen_kappa"],\n\n        **ci,\n        **latency,\n    }\n\n    # Per-image test predictions. test_loader is not shuffled, so the\n    # prediction order matches test_loader.dataset.base.samples.\n    flat_test_base = test_loader.dataset.base\n    prediction_rows = []\n\n    for idx, (yt, yp, conf) in enumerate(\n        zip(y_true, y_pred, confidence)\n    ):\n        sample_path = flat_test_base.samples[idx][0]\n\n        prediction_rows.append(\n            {\n                "sample_index": idx,\n                "path": sample_path,\n                "true_global_idx": int(yt),\n                "true_class": class_names[yt],\n                "pred_global_idx": int(yp),\n                "pred_class": class_names[yp],\n                "confidence": float(conf),\n                "correct": int(yt == yp),\n            }\n        )\n\n    pd.DataFrame(prediction_rows).to_csv(\n        PREDICTION_ROOT / f"{exp_name}_test_predictions.csv",\n        index=False,\n    )\n\n    # Save raw outputs\n    (\n        exp_dir / "history.json"\n    ).write_text(\n        json.dumps(history, indent=2)\n    )\n\n    save_history_csv(\n        history,\n        TABLE_ROOT / f"{exp_name}_history.csv",\n    )\n\n    (\n        exp_dir / "result.json"\n    ).write_text(\n        json.dumps(result, indent=2)\n    )\n\n    (\n        exp_dir / "classification_report.json"\n    ).write_text(\n        json.dumps(report, indent=2)\n    )\n\n    np.save(\n        exp_dir / "confusion_matrix.npy",\n        cm,\n    )\n\n    # 300 DPI figures\n    epochs = [\n        h["epoch"] for h in history\n    ]\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["loss"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["loss"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: training and validation loss",\n        "Epoch",\n        "Loss",\n        FIG_ROOT / f"{exp_name}_loss_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["accuracy"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["accuracy"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: training and validation accuracy",\n        "Epoch",\n        "Accuracy",\n        FIG_ROOT / f"{exp_name}_accuracy_curve.png",\n    )\n\n    plot_confusion_matrix(\n        cm,\n        class_names,\n        f"{exp_name}: test confusion matrix",\n        FIG_ROOT / f"{exp_name}_confusion_matrix_counts.png",\n        normalize=False,\n    )\n\n    plot_confusion_matrix(\n        cm,\n        class_names,\n        f"{exp_name}: normalized test confusion matrix",\n        FIG_ROOT / f"{exp_name}_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    metrics_df = plot_per_class_metrics(\n        report,\n        class_names,\n        f"{exp_name}: per-class test metrics",\n        FIG_ROOT / f"{exp_name}_per_class_metrics.png",\n    )\n\n    metrics_df.to_csv(\n        TABLE_ROOT\n        / f"{exp_name}_per_class_metrics.csv",\n        index=False,\n    )\n\n    plot_confidence_distribution(\n        y_true,\n        y_pred,\n        confidence,\n        f"{exp_name}: test confidence distribution",\n        FIG_ROOT\n        / f"{exp_name}_confidence_distribution.png",\n    )\n\n    print("TEST:", result)\n\n    del (\n        model,\n        train_loader,\n        val_loader,\n        test_loader,\n    )\n\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    return result\n\n\n\n\n# ============================================================\n# 17. HIERARCHICAL EXPERIMENT\n# ============================================================\n\ndef train_hier_experiment(\n    exp_name,\n    backbone_key,\n    variant,\n):\n    print("\\n" + "=" * 90)\n    print("EXPERIMENT:", exp_name)\n    print("TYPE: Hierarchical")\n    print("BACKBONE:", backbone_key)\n    print("VARIANT:", variant)\n    print("=" * 90)\n\n    seed_everything(SEED)\n\n    exp_dir = OUTPUT_ROOT / exp_name\n    exp_dir.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n\n    (\n        train_loader,\n        val_loader,\n        test_loader,\n        train_ds,\n        val_ds,\n        test_ds,\n        crop_class_counts,\n    ) = build_hier_loaders(backbone_key)\n\n    model = HierarchicalClassifier(\n        backbone_key,\n        crop_class_counts,\n        variant=variant,\n    )\n\n    model = maybe_parallel(model)\n\n    optimizer = make_fast_adamw(\n        model\n    )\n\n    scheduler = (\n        torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer,\n            T_max=MAX_EPOCHS,\n        )\n    )\n\n    scaler = torch.cuda.amp.GradScaler(\n        enabled=USE_AMP\n    )\n\n    best_val = float("inf")\n    best_epoch = 0\n    patience_left = PATIENCE\n\n    history = []\n\n    ckpt = (\n        CHECKPOINT_ROOT\n        / f"{exp_name}_best.pth"\n    )\n\n    start = time.time()\n\n    for epoch in range(1, MAX_EPOCHS + 1):\n        train_metrics = run_hier_epoch(\n            model,\n            train_loader,\n            optimizer,\n            scaler,\n        )\n\n        val_metrics = run_hier_epoch(\n            model,\n            val_loader,\n        )\n\n        scheduler.step()\n\n        history.append(\n            {\n                "epoch": epoch,\n                "lr": optimizer.param_groups[0]["lr"],\n                "train": train_metrics,\n                "val": val_metrics,\n            }\n        )\n\n        print(\n            f"[{epoch:03d}/{MAX_EPOCHS}] "\n            f"train_loss={train_metrics[\'loss\']:.4f} "\n            f"crop_acc={train_metrics[\'crop_accuracy\']:.4f} "\n            f"e2e_acc={train_metrics[\'end_to_end_accuracy\']:.4f} | "\n            f"val_loss={val_metrics[\'loss\']:.4f} "\n            f"crop_acc={val_metrics[\'crop_accuracy\']:.4f} "\n            f"e2e_acc={val_metrics[\'end_to_end_accuracy\']:.4f}"\n        )\n\n        if (\n            best_val - val_metrics["loss"]\n            > MIN_DELTA\n        ):\n            best_val = val_metrics["loss"]\n            best_epoch = epoch\n            patience_left = PATIENCE\n\n            save_state_dict(\n                model,\n                ckpt,\n            )\n\n        else:\n            patience_left -= 1\n\n        if patience_left <= 0:\n            print(\n                f"Early stopping at epoch {epoch}."\n            )\n            break\n\n    train_seconds = time.time() - start\n\n    unwrap_model(model).load_state_dict(\n        torch.load(\n            ckpt,\n            map_location=DEVICE,\n        )\n    )\n\n    test_result = evaluate_hierarchical_test(\n        model,\n        test_loader,\n        test_ds,\n    )\n\n    ci = bootstrap_ci(\n        test_result["global_true"],\n        test_result["global_pred"],\n        n_classes=len(test_ds.global_classes),\n    )\n\n    latency = benchmark_t4_latency(model)\n\n    result = {\n        "experiment": exp_name,\n        "family": "hierarchical",\n        "backbone": backbone_key,\n        "variant": variant,\n\n        "best_epoch": best_epoch,\n        "training_seconds": train_seconds,\n\n        "parameters": count_parameters(model),\n        "checkpoint_mb": checkpoint_size_mb(ckpt),\n\n        "test_stage1_accuracy":\n            test_result["crop"]["accuracy"],\n        "test_stage1_macro_f1":\n            test_result["crop"]["macro_f1"],\n\n        "test_potato_oracle_accuracy":\n            test_result["oracle_per_crop"]["potato"]["accuracy"],\n        "test_potato_oracle_macro_f1":\n            test_result["oracle_per_crop"]["potato"]["macro_f1"],\n\n        "test_rice_oracle_accuracy":\n            test_result["oracle_per_crop"]["rice"]["accuracy"],\n        "test_rice_oracle_macro_f1":\n            test_result["oracle_per_crop"]["rice"]["macro_f1"],\n\n        "test_tomato_oracle_accuracy":\n            test_result["oracle_per_crop"]["tomato"]["accuracy"],\n        "test_tomato_oracle_macro_f1":\n            test_result["oracle_per_crop"]["tomato"]["macro_f1"],\n\n        "test_accuracy":\n            test_result["end_to_end"]["accuracy"],\n        "test_balanced_accuracy":\n            test_result["end_to_end"]["balanced_accuracy"],\n        "test_macro_precision":\n            test_result["end_to_end"]["macro_precision"],\n        "test_macro_recall":\n            test_result["end_to_end"]["macro_recall"],\n        "test_macro_f1":\n            test_result["end_to_end"]["macro_f1"],\n        "test_weighted_f1":\n            test_result["end_to_end"]["weighted_f1"],\n        "test_mcc":\n            test_result["end_to_end"]["mcc"],\n        "test_cohen_kappa":\n            test_result["end_to_end"]["cohen_kappa"],\n\n        **ci,\n        **latency,\n    }\n\n    # Save every held-out test prediction to support paired statistical tests\n    # and manual error analysis.\n    prediction_rows = []\n\n    for idx, (yt, yp, conf) in enumerate(\n        zip(\n            test_result["global_true"],\n            test_result["global_pred"],\n            test_result["end_confidences"],\n        )\n    ):\n        sample_path = test_ds.samples[idx][0]\n\n        prediction_rows.append(\n            {\n                "sample_index": idx,\n                "path": sample_path,\n                "true_global_idx": int(yt),\n                "true_class": test_ds.global_classes[yt],\n                "pred_global_idx": int(yp),\n                "pred_class": test_ds.global_classes[yp],\n                "confidence": float(conf),\n                "correct": int(yt == yp),\n            }\n        )\n\n    pd.DataFrame(prediction_rows).to_csv(\n        PREDICTION_ROOT / f"{exp_name}_test_predictions.csv",\n        index=False,\n    )\n\n    # Save raw results\n    (\n        exp_dir / "history.json"\n    ).write_text(\n        json.dumps(history, indent=2)\n    )\n\n    save_history_csv(\n        history,\n        TABLE_ROOT / f"{exp_name}_history.csv",\n    )\n\n    (\n        exp_dir / "result.json"\n    ).write_text(\n        json.dumps(result, indent=2)\n    )\n\n    (\n        exp_dir / "stage1_report.json"\n    ).write_text(\n        json.dumps(\n            test_result["crop_report"],\n            indent=2,\n        )\n    )\n\n    (\n        exp_dir / "end_to_end_report.json"\n    ).write_text(\n        json.dumps(\n            test_result["global_report"],\n            indent=2,\n        )\n    )\n\n    for crop in CROPS:\n        (\n            exp_dir\n            / f"stage2_{crop}_report.json"\n        ).write_text(\n            json.dumps(\n                test_result[\n                    "per_crop_report"\n                ][crop],\n                indent=2,\n            )\n        )\n\n    np.save(\n        exp_dir / "stage1_crop_cm.npy",\n        test_result["crop_cm"],\n    )\n\n    np.save(\n        exp_dir / "end_to_end_cm.npy",\n        test_result["global_cm"],\n    )\n\n    for crop in CROPS:\n        np.save(\n            exp_dir / f"stage2_{crop}_cm.npy",\n            test_result["per_crop_cm"][crop],\n        )\n\n    # --------------------------------------------\n    # Figures for ALL hierarchical experiments\n    # --------------------------------------------\n    epochs = [\n        h["epoch"] for h in history\n    ]\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["loss"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["loss"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: total hierarchical loss",\n        "Epoch",\n        "Loss",\n        FIG_ROOT / f"{exp_name}_total_loss_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training crop loss": [\n                h["train"]["crop_loss"]\n                for h in history\n            ],\n            "Validation crop loss": [\n                h["val"]["crop_loss"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: crop-routing loss",\n        "Epoch",\n        "Crop loss",\n        FIG_ROOT / f"{exp_name}_crop_loss_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training disease loss": [\n                h["train"]["disease_loss"]\n                for h in history\n            ],\n            "Validation disease loss": [\n                h["val"]["disease_loss"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: disease-classification loss",\n        "Epoch",\n        "Disease loss",\n        FIG_ROOT / f"{exp_name}_disease_loss_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["crop_accuracy"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["crop_accuracy"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: Stage-1 crop-router accuracy",\n        "Epoch",\n        "Accuracy",\n        FIG_ROOT / f"{exp_name}_stage1_accuracy_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["end_to_end_accuracy"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["end_to_end_accuracy"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: true end-to-end accuracy",\n        "Epoch",\n        "Accuracy",\n        FIG_ROOT / f"{exp_name}_end_to_end_accuracy_curve.png",\n    )\n\n    save_line_plot(\n        epochs,\n        {\n            "Training": [\n                h["train"]["end_to_end_macro_f1"]\n                for h in history\n            ],\n            "Validation": [\n                h["val"]["end_to_end_macro_f1"]\n                for h in history\n            ],\n        },\n        f"{exp_name}: true end-to-end Macro-F1",\n        "Epoch",\n        "Macro-F1",\n        FIG_ROOT / f"{exp_name}_end_to_end_macro_f1_curve.png",\n    )\n\n    # Crop-specific Stage-2 accuracy curves\n    for crop in CROPS:\n        key = f"{crop}_accuracy"\n\n        save_line_plot(\n            epochs,\n            {\n                "Training": [\n                    h["train"][key]\n                    for h in history\n                ],\n                "Validation": [\n                    h["val"][key]\n                    for h in history\n                ],\n            },\n            f"{exp_name}: Stage-2 {crop.title()} disease accuracy",\n            "Epoch",\n            "Accuracy",\n            FIG_ROOT\n            / f"{exp_name}_stage2_{crop}_accuracy_curve.png",\n        )\n\n    # Stage-1 confusion matrices\n    plot_confusion_matrix(\n        test_result["crop_cm"],\n        [c.title() for c in CROPS],\n        f"{exp_name}: Stage-1 crop-router confusion matrix",\n        FIG_ROOT\n        / f"{exp_name}_stage1_confusion_matrix_counts.png",\n        normalize=False,\n    )\n\n    plot_confusion_matrix(\n        test_result["crop_cm"],\n        [c.title() for c in CROPS],\n        f"{exp_name}: normalized Stage-1 crop-router confusion matrix",\n        FIG_ROOT\n        / f"{exp_name}_stage1_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    # Stage-2 crop-wise confusion matrices + metrics\n    for crop in CROPS:\n        labels = test_ds.crop_classes[crop]\n        cm = test_result["per_crop_cm"][crop]\n        report = test_result["per_crop_report"][crop]\n\n        plot_confusion_matrix(\n            cm,\n            labels,\n            f"{exp_name}: Stage-2 {crop.title()} confusion matrix",\n            FIG_ROOT\n            / f"{exp_name}_stage2_{crop}_confusion_matrix_counts.png",\n            normalize=False,\n        )\n\n        plot_confusion_matrix(\n            cm,\n            labels,\n            f"{exp_name}: normalized Stage-2 {crop.title()} confusion matrix",\n            FIG_ROOT\n            / f"{exp_name}_stage2_{crop}_confusion_matrix_normalized.png",\n            normalize=True,\n        )\n\n        metric_df = plot_per_class_metrics(\n            report,\n            labels,\n            f"{exp_name}: Stage-2 {crop.title()} per-class metrics",\n            FIG_ROOT\n            / f"{exp_name}_stage2_{crop}_per_class_metrics.png",\n        )\n\n        metric_df.to_csv(\n            TABLE_ROOT\n            / f"{exp_name}_stage2_{crop}_per_class_metrics.csv",\n            index=False,\n        )\n\n    # End-to-end confusion matrix\n    plot_confusion_matrix(\n        test_result["global_cm"],\n        test_ds.global_classes,\n        f"{exp_name}: true end-to-end confusion matrix",\n        FIG_ROOT\n        / f"{exp_name}_end_to_end_confusion_matrix_counts.png",\n        normalize=False,\n    )\n\n    plot_confusion_matrix(\n        test_result["global_cm"],\n        test_ds.global_classes,\n        f"{exp_name}: normalized true end-to-end confusion matrix",\n        FIG_ROOT\n        / f"{exp_name}_end_to_end_confusion_matrix_normalized.png",\n        normalize=True,\n    )\n\n    # End-to-end class metrics\n    e2e_metric_df = plot_per_class_metrics(\n        test_result["global_report"],\n        test_ds.global_classes,\n        f"{exp_name}: true end-to-end per-class metrics",\n        FIG_ROOT\n        / f"{exp_name}_end_to_end_per_class_metrics.png",\n    )\n\n    e2e_metric_df.to_csv(\n        TABLE_ROOT\n        / f"{exp_name}_end_to_end_per_class_metrics.csv",\n        index=False,\n    )\n\n    # Confidence distribution\n    plot_confidence_distribution(\n        test_result["global_true"],\n        test_result["global_pred"],\n        test_result["end_confidences"],\n        f"{exp_name}: true end-to-end confidence distribution",\n        FIG_ROOT\n        / f"{exp_name}_end_to_end_confidence_distribution.png",\n    )\n\n    # Misclassification table\n    mis_rows = []\n\n    for yt, yp, conf in zip(\n        test_result["global_true"],\n        test_result["global_pred"],\n        test_result["end_confidences"],\n    ):\n        if yt != yp:\n            mis_rows.append(\n                {\n                    "true_class":\n                        test_ds.global_classes[yt],\n                    "predicted_class":\n                        test_ds.global_classes[yp],\n                    "confidence":\n                        conf,\n                }\n            )\n\n    pd.DataFrame(mis_rows).to_csv(\n        TABLE_ROOT\n        / f"{exp_name}_misclassifications.csv",\n        index=False,\n    )\n\n    print("TEST:", result)\n\n    del (\n        model,\n        train_loader,\n        val_loader,\n        test_loader,\n        train_ds,\n        val_ds,\n        test_ds,\n    )\n\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    return result\n\n\n\n\n\n# ============================================================\n# SINGLE-GPU WORKER ENTRY POINT\n# ============================================================\n\ndef run_memory_preflight():\n    """\n    Test EfficientNet-B3 with one AMP forward/backward/optimizer step at the\n    requested per-model batch size. If BS256 OOMs, the orchestrator uses the\n    same BS128 fallback for every model.\n    """\n    print("\\n" + "=" * 90)\n    print("BATCH-SIZE MEMORY PREFLIGHT")\n    print("=" * 90)\n    print("Requested batch size:", BATCH_SIZE)\n    print("Visible GPU:", torch.cuda.get_device_name(0))\n\n    try:\n        model = FlatClassifier(\n            "efficientnet_b3",\n            n_classes=16,\n        )\n        model = maybe_parallel(\n            model\n        )\n\n        optimizer = make_fast_adamw(\n            model\n        )\n\n        scaler = torch.cuda.amp.GradScaler(\n            enabled=USE_AMP\n        )\n\n        model.train()\n\n        images = torch.randn(\n            BATCH_SIZE,\n            3,\n            IMG_SIZE,\n            IMG_SIZE,\n            device=DEVICE,\n        )\n\n        targets = torch.randint(\n            0,\n            16,\n            (BATCH_SIZE,),\n            device=DEVICE,\n        )\n\n        optimizer.zero_grad(\n            set_to_none=True\n        )\n\n        with torch.cuda.amp.autocast(\n            enabled=USE_AMP\n        ):\n            logits = model(\n                images\n            )\n\n            loss = flat_ce(\n                logits,\n                targets,\n            )\n\n        scaler.scale(\n            loss\n        ).backward()\n\n        scaler.step(\n            optimizer\n        )\n\n        scaler.update()\n\n        torch.cuda.synchronize()\n\n        peak_gib = (\n            torch.cuda.max_memory_allocated()\n            / (\n                1024 ** 3\n            )\n        )\n\n        print(\n            f"PREFLIGHT PASS: batch={BATCH_SIZE}, "\n            f"peak allocated={peak_gib:.2f} GiB"\n        )\n\n        return 0\n\n    except torch.cuda.OutOfMemoryError:\n        print(\n            f"PREFLIGHT OOM: batch={BATCH_SIZE} "\n            "does not fit safely on one T4."\n        )\n\n        return 42\n\n\ndef run_fasternet_interface_preflight():\n    """\n    Verify the exact FasterNet-M feature width before any long experiment.\n    No training is performed.\n    """\n    print("\\n" + "=" * 90)\n    print("FASTERNET FEATURE-INTERFACE PREFLIGHT")\n    print("=" * 90)\n\n    model, feat_dim = create_backbone(\n        "fasternet_m"\n    )\n\n    model = model.to(\n        DEVICE\n    )\n\n    model.eval()\n\n    with torch.inference_mode():\n        x = torch.randn(\n            2,\n            3,\n            IMG_SIZE,\n            IMG_SIZE,\n            device=DEVICE,\n        )\n\n        y = model(\n            x\n        )\n\n    print(\n        "Resolved FasterNet:",\n        RESOLVED_BACKBONES["fasternet_m"],\n    )\n    print(\n        "Declared feature dim:",\n        feat_dim,\n    )\n    print(\n        "Actual pooled feature shape:",\n        tuple(\n            y.shape\n        ),\n    )\n\n    if (\n        y.ndim != 2\n        or y.shape[1] != feat_dim\n    ):\n        raise RuntimeError(\n            "FasterNet interface preflight failed."\n        )\n\n    print(\n        "FASTERNET FEATURE INTERFACE: PASS"\n    )\n\n\ndef run_selected_experiment():\n    exp_name = os.environ[\n        "LEAFCARENET_EXPERIMENT"\n    ]\n\n    exp_lookup = {\n        name: (\n            family,\n            backbone_key,\n            variant,\n        )\n        for (\n            name,\n            family,\n            backbone_key,\n            variant,\n        ) in EXPERIMENTS\n    }\n\n    if exp_name not in exp_lookup:\n        raise KeyError(\n            f"Unknown experiment \'{exp_name}\'."\n        )\n\n    (\n        family,\n        backbone_key,\n        variant,\n    ) = exp_lookup[\n        exp_name\n    ]\n\n    print("\\n" + "#" * 100)\n    print("PARALLEL SINGLE-GPU WORKER")\n    print("Experiment:", exp_name)\n    print(\n        "Visible GPU:",\n        torch.cuda.get_device_name(0),\n    )\n    print(\n        "Train batch:",\n        BATCH_SIZE,\n    )\n    print(\n        "Eval batch:",\n        EVAL_BATCH_SIZE,\n    )\n    print(\n        "DataLoader workers:",\n        NUM_WORKERS,\n    )\n    print(\n        "Online augmentation: DISABLED"\n    )\n    print("#" * 100)\n\n    write_experiment_config(\n        exp_name,\n        family,\n        backbone_key,\n        variant,\n    )\n\n    if family == "flat":\n        result = train_flat_experiment(\n            exp_name,\n            backbone_key,\n        )\n    else:\n        result = train_hier_experiment(\n            exp_name,\n            backbone_key,\n            variant,\n        )\n\n    completion = {\n        "experiment": exp_name,\n        "batch_size": BATCH_SIZE,\n        "eval_batch_size": EVAL_BATCH_SIZE,\n        "num_workers": NUM_WORKERS,\n        "online_augmentation": False,\n        "gpu": torch.cuda.get_device_name(0),\n        "result": result,\n    }\n\n    (\n        OUTPUT_ROOT\n        / "WORKER_COMPLETED.json"\n    ).write_text(\n        json.dumps(\n            completion,\n            indent=2,\n        )\n    )\n\n    print(\n        "\\nWORKER COMPLETED:",\n        exp_name,\n    )\n\n\nif __name__ == "__main__":\n    mode = os.environ.get(\n        "LEAFCARENET_MODE",\n        "train",\n    )\n\n    if mode == "memory_preflight":\n        sys.exit(\n            run_memory_preflight()\n        )\n\n    if mode == "fasternet_preflight":\n        run_fasternet_interface_preflight()\n        sys.exit(0)\n\n    run_selected_experiment()\n'

WORKER_SCRIPT_PATH.write_text(
    WORKER_SCRIPT,
    encoding="utf-8",
)

print("Worker written:", WORKER_SCRIPT_PATH)
print("Worker size:", WORKER_SCRIPT_PATH.stat().st_size, "bytes")
print("Fast path: shared uint8 spatial cache + GPU-side normalization")


Worker written: /kaggle/working/leafcarenet_journal8_worker.py
Worker size: 107671 bytes
Fast path: shared uint8 spatial cache + GPU-side normalization


In [7]:

# ============================================================
# 3. RESTORE COMPLETED NEW/LEGACY ZIPS + VALIDATE + PRE-CACHE
# ============================================================

import hashlib

def model_root(exp_name):
    return BASE_OUTPUT / exp_name


def completion_file(exp_name):
    return model_root(exp_name) / "WORKER_COMPLETED.json"


def result_file(exp_name):
    return model_root(exp_name) / exp_name / "result.json"


def prediction_file(exp_name):
    return model_root(exp_name) / "predictions" / f"{exp_name}_test_predictions.csv"


def is_complete(exp_name):
    """
    Canonical completion check used by the scheduler.
    Requires:
      1) canonical completion marker
      2) result.json
      3) paired test predictions for all 916 held-out images
    """
    c = completion_file(exp_name)
    r = result_file(exp_name)
    p = prediction_file(exp_name)

    if not (c.exists() and r.exists() and p.exists()):
        return False

    try:
        payload = json.loads(c.read_text(encoding="utf-8"))
        result = json.loads(r.read_text(encoding="utf-8"))
        pred = pd.read_csv(p)

        if payload.get("experiment") != exp_name:
            return False
        if result.get("experiment") != exp_name:
            return False
        if len(pred) != 916:
            return False

        required_cols = {"sample_index", "path", "correct"}
        if not required_cols.issubset(pred.columns):
            return False

        return True

    except Exception:
        return False


def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(
                    f"Unsafe path in ZIP {zip_path}: {member.filename}"
                )
        zf.extractall(destination)


# ------------------------------------------------------------------
# A. New-format JOURNAL8 bundles
# ------------------------------------------------------------------

def restore_new_format_bundles():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []

    restored = []

    zip_candidates = sorted(
        input_root.rglob("JOURNAL8_*_COMPLETE.zip")
    )

    if zip_candidates:
        print("\nNEW-FORMAT RESUME BUNDLES FOUND:", len(zip_candidates))

    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                names = set(zf.namelist())

                if "RESUME_BUNDLE_MANIFEST.json" not in names:
                    continue

                manifest = json.loads(
                    zf.read("RESUME_BUNDLE_MANIFEST.json").decode("utf-8")
                )

            exp_name = manifest.get("experiment")

            if exp_name not in MODEL_META:
                print("Skipping unknown JOURNAL8 bundle:", exp_name)
                continue

            if is_complete(exp_name):
                continue

            dest = model_root(exp_name)

            if dest.exists():
                shutil.rmtree(dest)

            safe_extract_zip(
                zip_path,
                dest,
            )

            if not is_complete(exp_name):
                raise RuntimeError(
                    f"Restored bundle failed validation: {zip_path}"
                )

            restored.append(exp_name)
            print(
                "RESTORED NEW FORMAT:",
                exp_name,
                "<-",
                zip_path,
            )

        except Exception as exc:
            print(
                "New-format restore warning:",
                zip_path,
                "->",
                repr(exc),
            )

    return restored


# ------------------------------------------------------------------
# B. Legacy experiment mappings
# ------------------------------------------------------------------

LEGACY_TO_FINAL = {
    # Canonical/new IDs are also accepted.
    "01_flat_mobilenetv2": "01_flat_mobilenetv2",
    "02_flat_efficientnet_b3": "02_flat_efficientnet_b3",
    "03_flat_fasternet_m": "03_flat_fasternet_m",
    "04_flat_deit_tiny": "04_flat_deit_tiny",
    "05_leafcarenet": "05_leafcarenet",
    "06_hier_efficientnet_b3": "06_hier_efficientnet_b3",
    "07_hier_fasternet_m": "07_hier_fasternet_m",
    "08_hier_deit_tiny": "08_hier_deit_tiny",

    # Previous notebook IDs.
    "09_hier_mobilenetv2_linear": "05_leafcarenet",
    "10_hier_efficientnet_b3_linear": "06_hier_efficientnet_b3",
    "06_hier_fasternet_m_linear": "07_hier_fasternet_m",
}

INTENTIONALLY_IGNORED_LEGACY = {
    "07_hier_fasternet_m_se":
        "FasterNet-M Hierarchical + SE was removed from the final benchmark.",
    "08_leafcarenet_full":
        "Old FasterNet-based LeafCareNet was removed; final LeafCareNet is linear Hierarchical MobileNetV2.",
}

EXPECTED_SCHEMA = {
    "01_flat_mobilenetv2": {
        "family": "flat",
        "backbone": "mobilenetv2",
        "variant": "flat_16class",
    },
    "02_flat_efficientnet_b3": {
        "family": "flat",
        "backbone": "efficientnet_b3",
        "variant": "flat_16class",
    },
    "03_flat_fasternet_m": {
        "family": "flat",
        "backbone": "fasternet_m",
        "variant": "flat_16class",
    },
    "04_flat_deit_tiny": {
        "family": "flat",
        "backbone": "deit_tiny",
        "variant": "flat_16class",
    },
    "05_leafcarenet": {
        "family": "hierarchical",
        "backbone": "mobilenetv2",
        "variant": "linear",
    },
    "06_hier_efficientnet_b3": {
        "family": "hierarchical",
        "backbone": "efficientnet_b3",
        "variant": "linear",
    },
    "07_hier_fasternet_m": {
        "family": "hierarchical",
        "backbone": "fasternet_m",
        "variant": "linear",
    },
    "08_hier_deit_tiny": {
        "family": "hierarchical",
        "backbone": "deit_tiny",
        "variant": "linear",
    },
}


def find_output_root(result_path):
    """
    In the old workers, result.json normally lived at:
        OUTPUT_ROOT / experiment / result.json

    OUTPUT_ROOT then also contained predictions/, checkpoints/, tables/, etc.
    """
    result_path = Path(result_path)

    candidates = [
        result_path.parent.parent,
        result_path.parent,
    ]

    for candidate in candidates:
        if (
            (candidate / "predictions").exists()
            or (candidate / "checkpoints").exists()
            or (candidate / "WORKER_COMPLETED.json").exists()
            or (candidate / "figures_300dpi").exists()
        ):
            return candidate

    return result_path.parent.parent


def find_named_resource(root, exact_name, suffix_pattern=None):
    root = Path(root)

    exact = list(root.rglob(exact_name))
    if exact:
        return exact[0]

    if suffix_pattern:
        matches = list(root.rglob(suffix_pattern))
        if matches:
            return matches[0]

    return None


def load_matching_completion(root, legacy_exp):
    root = Path(root)

    for path in root.rglob("WORKER_COMPLETED.json"):
        try:
            payload = json.loads(
                path.read_text(encoding="utf-8")
            )
            if payload.get("experiment") == legacy_exp:
                return payload, path
        except Exception:
            pass

    return None, None


def validate_legacy_candidate(result_path, source_label):
    """
    Returns a candidate dict or None.

    A model is considered reusable only when:
      - result schema matches the FINAL model definition
      - held-out prediction CSV exists
      - prediction CSV contains exactly 916 test samples
      - known metadata does not indicate online augmentation or incompatible batch
    """
    result_path = Path(result_path)

    try:
        result = json.loads(
            result_path.read_text(encoding="utf-8")
        )
    except Exception:
        return None

    legacy_exp = result.get("experiment") or result_path.parent.name

    if legacy_exp in INTENTIONALLY_IGNORED_LEGACY:
        return {
            "status": "ignored",
            "legacy_experiment": legacy_exp,
            "reason": INTENTIONALLY_IGNORED_LEGACY[legacy_exp],
            "source": str(source_label),
        }

    final_exp = LEGACY_TO_FINAL.get(legacy_exp)

    if final_exp is None:
        return None

    expected = EXPECTED_SCHEMA[final_exp]

    # Architecture/result schema validation.
    for key in ["family", "backbone", "variant"]:
        if result.get(key) != expected[key]:
            return {
                "status": "rejected",
                "legacy_experiment": legacy_exp,
                "final_experiment": final_exp,
                "reason": (
                    f"{key} mismatch: got {result.get(key)!r}, "
                    f"expected {expected[key]!r}"
                ),
                "source": str(source_label),
            }

    output_root = find_output_root(result_path)

    pred = find_named_resource(
        output_root,
        f"{legacy_exp}_test_predictions.csv",
        "*_test_predictions.csv",
    )

    if pred is None:
        return {
            "status": "rejected",
            "legacy_experiment": legacy_exp,
            "final_experiment": final_exp,
            "reason": "test prediction CSV not found",
            "source": str(source_label),
        }

    try:
        pred_df = pd.read_csv(pred)
    except Exception as exc:
        return {
            "status": "rejected",
            "legacy_experiment": legacy_exp,
            "final_experiment": final_exp,
            "reason": f"prediction CSV unreadable: {exc}",
            "source": str(source_label),
        }

    required_cols = {"sample_index", "path", "correct"}

    if len(pred_df) != 916:
        return {
            "status": "rejected",
            "legacy_experiment": legacy_exp,
            "final_experiment": final_exp,
            "reason": f"test predictions = {len(pred_df)}, expected 916",
            "source": str(source_label),
        }

    if not required_cols.issubset(pred_df.columns):
        return {
            "status": "rejected",
            "legacy_experiment": legacy_exp,
            "final_experiment": final_exp,
            "reason": f"prediction columns missing: {sorted(required_cols - set(pred_df.columns))}",
            "source": str(source_label),
        }

    completion, completion_path = load_matching_completion(
        output_root,
        legacy_exp,
    )

    # If old metadata explicitly says augmentation was used, do not mix it
    # with the final no-augmentation benchmark.
    if completion is not None:
        if completion.get("online_augmentation") is True:
            return {
                "status": "rejected",
                "legacy_experiment": legacy_exp,
                "final_experiment": final_exp,
                "reason": "legacy metadata says online_augmentation=True",
                "source": str(source_label),
            }

        old_batch = completion.get("batch_size")

        if old_batch is not None and int(old_batch) != TRAIN_BATCH_SIZE:
            return {
                "status": "rejected",
                "legacy_experiment": legacy_exp,
                "final_experiment": final_exp,
                "reason": (
                    f"legacy train batch={old_batch}, "
                    f"final common batch={TRAIN_BATCH_SIZE}"
                ),
                "source": str(source_label),
            }

    checkpoint = find_named_resource(
        output_root,
        f"{legacy_exp}_best.pth",
        "*_best.pth",
    )

    history = find_named_resource(
        output_root,
        "history.json",
        "*_history.csv",
    )

    fig_dir = output_root / "figures_300dpi"

    score = 100  # valid result schema
    score += 60  # valid 916-image paired predictions
    score += 30 if checkpoint is not None else 0
    score += 15 if completion is not None else 0
    score += 5 if history is not None else 0
    score += 5 if fig_dir.exists() else 0

    return {
        "status": "valid",
        "legacy_experiment": legacy_exp,
        "final_experiment": final_exp,
        "result_path": result_path,
        "output_root": output_root,
        "prediction_path": pred,
        "checkpoint_path": checkpoint,
        "completion": completion,
        "completion_path": completion_path,
        "score": score,
        "source": str(source_label),
    }


# ------------------------------------------------------------------
# C. Discover legacy results from normal folders and arbitrary ZIP names
# ------------------------------------------------------------------

LEGACY_SCAN_ROOT = Path(
    "/kaggle/working/_legacy_result_scan"
)

if LEGACY_SCAN_ROOT.exists():
    shutil.rmtree(LEGACY_SCAN_ROOT)

LEGACY_SCAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def zip_may_contain_results(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = zf.namelist()

        return (
            any(
                Path(name).name == "result.json"
                for name in names
            )
            or any(
                name.lower().endswith(".zip")
                for name in names
            )
        )

    except Exception:
        return False


def extract_candidate_zip(zip_path, depth=0):
    """
    Extract only ZIPs that appear to contain result.json or nested ZIP files.
    Nested ZIPs are scanned up to depth 2.
    """
    if depth > 2:
        return []

    zip_path = Path(zip_path)

    digest = hashlib.sha1(
        str(zip_path).encode("utf-8")
    ).hexdigest()[:10]

    dest = (
        LEGACY_SCAN_ROOT
        / f"{zip_path.stem}_{digest}_d{depth}"
    )

    if dest.exists():
        shutil.rmtree(dest)

    safe_extract_zip(
        zip_path,
        dest,
    )

    roots = [dest]

    if depth < 2:
        for nested_zip in dest.rglob("*.zip"):
            if zip_may_contain_results(nested_zip):
                try:
                    roots.extend(
                        extract_candidate_zip(
                            nested_zip,
                            depth=depth + 1,
                        )
                    )
                except Exception as exc:
                    print(
                        "Nested ZIP scan warning:",
                        nested_zip,
                        "->",
                        repr(exc),
                    )

    return roots


def discover_legacy_candidates():
    input_root = Path("/kaggle/input")
    working_root = Path("/kaggle/working")

    # Previous compatible results may already be present in the same
    # persistent Kaggle draft session under /kaggle/working.
    search_roots = []

    if input_root.exists():
        search_roots.append(input_root)

    if working_root.exists():
        search_roots.append(working_root)

    # Only extract ZIPs attached through /kaggle/input.
    # Do not unpack large ZIPs already in /kaggle/working because their
    # raw result folders are normally already present there.
    zip_paths = sorted(
        input_root.rglob("*.zip")
    ) if input_root.exists() else []

    print(
        "\nLegacy raw-result scan roots:",
        [str(x) for x in search_roots],
    )

    print(
        "ALL ZIP FILES FOUND UNDER /kaggle/input:",
        len(zip_paths),
    )

    extracted_zip_count = 0

    for zip_path in zip_paths:
        # New JOURNAL8 bundles are already restored by the new-format loader.
        if zip_path.name.startswith("JOURNAL8_"):
            continue

        if not zip_may_contain_results(zip_path):
            continue

        try:
            roots = extract_candidate_zip(
                zip_path
            )
            search_roots.extend(roots)
            extracted_zip_count += 1
            print(
                "Legacy ZIP opened:",
                zip_path,
            )

        except Exception as exc:
            print(
                "Legacy ZIP extraction warning:",
                zip_path,
                "->",
                repr(exc),
            )

    print(
        "Legacy ZIPs selected for result scan:",
        extracted_zip_count,
    )

    candidates = []
    notices = []
    seen_result_paths = set()

    for root in search_roots:
        for result_path in root.rglob("result.json"):
            # Do not treat the current canonical output tree or temporary
            # extraction cache as a legacy source.
            try:
                resolved_result = result_path.resolve()

                if BASE_OUTPUT.resolve() in resolved_result.parents:
                    continue

                if LEGACY_SCAN_ROOT.resolve() in resolved_result.parents:
                    continue

            except Exception:
                pass

            key = str(
                result_path.resolve()
            )

            if key in seen_result_paths:
                continue

            seen_result_paths.add(key)

            candidate = validate_legacy_candidate(
                result_path,
                source_label=root,
            )

            if candidate is None:
                continue

            if candidate["status"] == "valid":
                candidates.append(candidate)
            else:
                notices.append(candidate)

    return candidates, notices


def canonicalize_legacy_candidate(candidate):
    """
    Copy the best legacy model output into the current canonical JOURNAL8 layout
    and rename experiment-specific files so the rest of the notebook can use it
    exactly like a newly trained model.
    """
    legacy_exp = candidate["legacy_experiment"]
    final_exp = candidate["final_experiment"]
    source_root = Path(candidate["output_root"])
    dest = model_root(final_exp)

    if dest.exists():
        shutil.rmtree(dest)

    shutil.copytree(
        source_root,
        dest,
    )

    # Rename directories/files containing the old experiment ID.
    if legacy_exp != final_exp:
        paths = sorted(
            dest.rglob("*"),
            key=lambda p: len(p.parts),
            reverse=True,
        )

        for path in paths:
            if legacy_exp not in path.name:
                continue

            new_name = path.name.replace(
                legacy_exp,
                final_exp,
            )

            new_path = path.with_name(
                new_name
            )

            if new_path.exists():
                if new_path.is_dir():
                    shutil.rmtree(new_path)
                else:
                    new_path.unlink()

            path.rename(
                new_path
            )

    # Ensure canonical nested result directory exists.
    canonical_result_dir = (
        dest
        / final_exp
    )

    canonical_result_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Locate result after copy/rename.
    result_candidates = list(
        dest.rglob("result.json")
    )

    if not result_candidates:
        raise RuntimeError(
            f"No result.json after restoring {legacy_exp}"
        )

    # Prefer canonical nested result.json if already there.
    restored_result = (
        canonical_result_dir
        / "result.json"
    )

    if restored_result.exists():
        source_result = restored_result
    else:
        source_result = result_candidates[0]
        shutil.copy2(
            source_result,
            restored_result,
        )

    result = json.loads(
        restored_result.read_text(
            encoding="utf-8"
        )
    )

    result[
        "restored_from_legacy_experiment"
    ] = legacy_exp

    result[
        "experiment"
    ] = final_exp

    restored_result.write_text(
        json.dumps(
            result,
            indent=2,
        ),
        encoding="utf-8",
    )

    # Ensure prediction file has the canonical experiment ID.
    pred_dir = (
        dest
        / "predictions"
    )

    pred_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    canonical_pred = (
        pred_dir
        / f"{final_exp}_test_predictions.csv"
    )

    if not canonical_pred.exists():
        pred_candidates = list(
            pred_dir.glob("*_test_predictions.csv")
        )

        if not pred_candidates:
            pred_candidates = list(
                dest.rglob("*_test_predictions.csv")
            )

        if not pred_candidates:
            raise RuntimeError(
                f"No test prediction CSV after restoring {legacy_exp}"
            )

        shutil.copy2(
            pred_candidates[0],
            canonical_pred,
        )

    pred_df = pd.read_csv(
        canonical_pred
    )

    if len(pred_df) != 916:
        raise RuntimeError(
            f"Restored prediction count for {final_exp} is "
            f"{len(pred_df)}, expected 916."
        )

    # Preserve any useful old completion metadata but write a clean canonical marker.
    old_completion = (
        candidate.get("completion")
        or {}
    )

    canonical_completion = {
        "experiment": final_exp,
        "restored_from_legacy": True,
        "legacy_experiment": legacy_exp,
        "legacy_source": candidate["source"],
        "batch_size": int(
            old_completion.get(
                "batch_size",
                TRAIN_BATCH_SIZE,
            )
        ),
        "eval_batch_size": int(
            old_completion.get(
                "eval_batch_size",
                EVAL_BATCH_SIZE,
            )
        ),
        "online_augmentation": bool(
            old_completion.get(
                "online_augmentation",
                False,
            )
        ),
        "dataset_counts": {
            "train": 13720,
            "val": 3660,
            "test": 916,
        },
        "result": result,
    }

    completion_file(
        final_exp
    ).write_text(
        json.dumps(
            canonical_completion,
            indent=2,
        ),
        encoding="utf-8",
    )

    restore_note = {
        "final_experiment": final_exp,
        "legacy_experiment": legacy_exp,
        "display_name": MODEL_META[final_exp]["display_name"],
        "source": candidate["source"],
        "candidate_score": candidate["score"],
        "checkpoint_available": candidate["checkpoint_path"] is not None,
        "restored_at": datetime.now().isoformat(),
    }

    (
        dest
        / "RESTORED_FROM_LEGACY.json"
    ).write_text(
        json.dumps(
            restore_note,
            indent=2,
        ),
        encoding="utf-8",
    )

    if not is_complete(final_exp):
        raise RuntimeError(
            f"Canonical validation failed after restoring {legacy_exp} "
            f"as {final_exp}"
        )

    return restore_note


# ------------------------------------------------------------------
# D. Execute restore
# ------------------------------------------------------------------

restored_new = restore_new_format_bundles()

legacy_candidates, legacy_notices = discover_legacy_candidates()

# Print intentionally ignored/rejected legacy candidates for transparency.
if legacy_notices:
    print("\nLEGACY FILES NOT USED:")
    for item in legacy_notices:
        print(
            " -",
            item.get("legacy_experiment"),
            "|",
            item.get("status"),
            "|",
            item.get("reason"),
        )

# Keep only the most complete valid copy for each final experiment.
best_by_final = {}

for candidate in legacy_candidates:
    final_exp = candidate["final_experiment"]

    if is_complete(final_exp):
        continue

    current = best_by_final.get(
        final_exp
    )

    if (
        current is None
        or candidate["score"] > current["score"]
    ):
        best_by_final[
            final_exp
        ] = candidate


restored_legacy = []

for final_exp in EXPERIMENT_ORDER:
    candidate = best_by_final.get(
        final_exp
    )

    if candidate is None:
        continue

    try:
        note = canonicalize_legacy_candidate(
            candidate
        )

        restored_legacy.append(
            final_exp
        )

        print(
            "\nRESTORED LEGACY MODEL:"
        )
        print(
            "  final   :",
            final_exp,
            "|",
            MODEL_META[final_exp]["display_name"],
        )
        print(
            "  old id  :",
            candidate["legacy_experiment"],
        )
        print(
            "  score   :",
            candidate["score"],
        )
        print(
            "  checkpoint:",
            "YES"
            if candidate["checkpoint_path"] is not None
            else "NO",
        )

    except Exception as exc:
        print(
            "Legacy canonicalization warning:",
            final_exp,
            "->",
            repr(exc),
        )


# ------------------------------------------------------------------
# E. Final resume audit table
# ------------------------------------------------------------------

audit_rows = []

for exp_name in EXPERIMENT_ORDER:
    root = model_root(
        exp_name
    )

    restored_note = (
        root
        / "RESTORED_FROM_LEGACY.json"
    )

    source = "new training / JOURNAL8 bundle"

    if restored_note.exists():
        try:
            note = json.loads(
                restored_note.read_text(
                    encoding="utf-8"
                )
            )
            source = (
                "legacy ZIP: "
                + note.get(
                    "legacy_experiment",
                    "?",
                )
            )
        except Exception:
            source = "legacy ZIP"

    pred_path = prediction_file(
        exp_name
    )

    checkpoint_count = len(
        list(
            root.rglob("*.pth")
        )
    ) if root.exists() else 0

    audit_rows.append(
        {
            "experiment": exp_name,
            "model": MODEL_META[exp_name]["display_name"],
            "status": "SKIP - VERIFIED COMPLETE"
                if is_complete(exp_name)
                else "TRAIN",
            "source": source
                if is_complete(exp_name)
                else "-",
            "test_predictions": (
                len(pd.read_csv(pred_path))
                if pred_path.exists()
                else 0
            ),
            "checkpoint_files": checkpoint_count,
        }
    )


resume_audit_df = pd.DataFrame(
    audit_rows
)

print(
    "\n"
    + "=" * 120
)
print(
    "FINAL RESUME AUDIT — THIS TABLE DECIDES WHAT WILL TRAIN"
)
print(
    "=" * 120
)
print(
    resume_audit_df.to_string(
        index=False
    )
)
print(
    "=" * 120
)

resume_audit_df.to_csv(
    "/kaggle/working/JOURNAL8_RESUME_AUDIT.csv",
    index=False,
)

completed_now = [
    e
    for e
    in EXPERIMENT_ORDER
    if is_complete(e)
]

pending_now = [
    e
    for e
    in EXPERIMENT_ORDER
    if not is_complete(e)
]

print(
    "\nVerified completed / will SKIP:",
    completed_now
)

print(
    "Pending / will TRAIN:",
    pending_now
)

# Temporary extraction cache is no longer required after canonical restore.
# Remove it so legacy ZIP scanning does not unnecessarily inflate /kaggle/working.
if LEGACY_SCAN_ROOT.exists():
    try:
        shutil.rmtree(
            LEGACY_SCAN_ROOT
        )
        print(
            "Temporary legacy ZIP scan cache removed:",
            LEGACY_SCAN_ROOT,
        )
    except Exception as exc:
        print(
            "Temporary scan-cache cleanup warning:",
            repr(exc),
        )


# ------------------------------------------------------------------
# F. Serial pre-cache ONLY the backbones still needed.
# This avoids wasting startup time on already completed models.
# ------------------------------------------------------------------

if pending_now:
    try:
        import timm
    except ImportError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "timm",
            ]
        )
        import timm

    BACKBONE_CANDIDATES = {
        "mobilenetv2": [
            "mobilenetv2_100.ra_in1k",
            "mobilenetv2_100",
        ],
        "efficientnet_b3": [
            "efficientnet_b3.ra2_in1k",
            "efficientnet_b3",
        ],
        "fasternet_m": [
            "fasternet_m.in1k",
            "fasternet_m",
        ],
        "deit_tiny": [
            "deit_tiny_patch16_224.fb_in1k",
            "deit_tiny_patch16_224",
        ],
    }

    try:
        available = set(
            timm.list_models(
                include_tags=True
            )
        )
    except TypeError:
        available = set(
            timm.list_models()
        )

    needed_backbones = sorted(
        {
            MODEL_META[exp]["backbone"]
            for exp
            in pending_now
        }
    )

    print(
        "\nBackbones still needed:",
        needed_backbones,
    )

    resolved = {}

    for key in needed_backbones:
        candidates = BACKBONE_CANDIDATES[
            key
        ]

        name = next(
            (
                x
                for x
                in candidates
                if x in available
            ),
            None,
        )

        if name is None:
            patterns = {
                "mobilenetv2": "*mobilenetv2*",
                "efficientnet_b3": "*efficientnet_b3*",
                "fasternet_m": "*fasternet_m*",
                "deit_tiny": "*deit_tiny_patch16_224*",
            }

            matches = timm.list_models(
                patterns[key]
            )

            if not matches:
                raise RuntimeError(
                    f"No timm model found for {key}"
                )

            name = matches[0]

        resolved[key] = name

    print(
        "\nPre-caching ONLY pending pretrained backbones:"
    )

    for key, name in resolved.items():
        print(
            "  loading",
            key,
            "->",
            name,
        )

        try:
            m = timm.create_model(
                name,
                pretrained=True,
                num_classes=0,
                global_pool="avg",
            )

            print(
                "    num_features =",
                getattr(
                    m,
                    "num_features",
                    "?",
                ),
            )

            del m
            gc.collect()

        except Exception as exc:
            raise RuntimeError(
                f"Could not cache pretrained {name}. Enable Internet in Kaggle "
                f"or attach the pretrained cache. Original error: {exc}"
            )

    print(
        "Pending-backbone cache warm-up: PASS"
    )



Legacy raw-result scan roots: ['/kaggle/input', '/kaggle/working']
ALL ZIP FILES FOUND UNDER /kaggle/input: 0
Legacy ZIPs selected for result scan: 0

LEGACY FILES NOT USED:
 - 08_leafcarenet_full | ignored | Old FasterNet-based LeafCareNet was removed; final LeafCareNet is linear Hierarchical MobileNetV2.
 - 07_hier_fasternet_m_se | ignored | FasterNet-M Hierarchical + SE was removed from the final benchmark.

FINAL RESUME AUDIT — THIS TABLE DECIDES WHAT WILL TRAIN
             experiment                        model                   status                                     source  test_predictions  checkpoint_files
    01_flat_mobilenetv2             MobileNetV2 Flat SKIP - VERIFIED COMPLETE            legacy ZIP: 01_flat_mobilenetv2               916                 1
02_flat_efficientnet_b3         EfficientNet-B3 Flat SKIP - VERIFIED COMPLETE        legacy ZIP: 02_flat_efficientnet_b3               916                 1
    03_flat_fasternet_m             FasterNet-M Flat SKIP

Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


    num_features = 192
Pending-backbone cache warm-up: PASS


In [8]:

# ============================================================
# 4. BUILD ONE SHARED FAST CACHE FOR PENDING DeiT MODELS
# ============================================================
#
# Why this is faster:
#   Old path (every epoch):
#     JPEG -> PIL decode -> resize/crop -> ToTensor -> normalize -> GPU
#
#   Fast path (one time only):
#     JPEG -> spatial preprocess -> uint8 cache
#
#   Every training epoch:
#     uint8 memmap -> pinned RAM -> GPU -> normalize on GPU
#
# The model, labels, sampler, batch size, optimizer, losses and train/val/test
# split are unchanged. This is an execution/data-loading optimization only.
# ============================================================

import hashlib
from tqdm.auto import tqdm
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torch.utils.data import Dataset, DataLoader
import timm
from timm.data import create_transform, resolve_model_data_config

FAST_CACHE_BACKBONES = {
    MODEL_META[e]["backbone"]
    for e in pending_now
    if MODEL_META[e]["backbone"] == "deit_tiny"
}

SPEED_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CROPS_FAST = [
    "potato",
    "rice",
    "tomato",
]

IMAGE_EXTS_FAST = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def _has_crop_structure_fast(root):
    root = Path(root)
    return all(
        (root / f"{crop}_leaf").is_dir()
        for crop in CROPS_FAST
    )


def _locate_dataset_fast():
    preferred = Path(
        "/kaggle/input/datasets/mahbubur03078/leafdisease2"
    )

    if preferred.exists():
        if _has_crop_structure_fast(preferred):
            return preferred

        candidates = []
        for p in preferred.rglob("potato_leaf"):
            if _has_crop_structure_fast(p.parent):
                candidates.append(p.parent)

        if candidates:
            return sorted(
                candidates,
                key=lambda x: len(x.parts),
            )[0]

    for base in [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]:
        if not base.exists():
            continue

        candidates = []

        for p in base.rglob("potato_leaf"):
            if _has_crop_structure_fast(p.parent):
                candidates.append(p.parent)

        if candidates:
            # Prefer a path containing leafdisease2.
            preferred_matches = [
                p
                for p in candidates
                if "leafdisease2" in str(p).lower()
            ]

            pool = (
                preferred_matches
                if preferred_matches
                else candidates
            )

            return sorted(
                pool,
                key=lambda x: len(x.parts),
            )[0]

    raise FileNotFoundError(
        "Could not locate potato_leaf/rice_leaf/tomato_leaf dataset."
    )


FAST_DATA_ROOT = _locate_dataset_fast()

print(
    "Fast-cache dataset root:",
    FAST_DATA_ROOT,
)


def _discover_samples_fast(root, split):
    root = Path(root)

    crop_classes = {}
    disease_to_idx = {}
    global_to_idx = {}
    global_classes = []
    samples = []

    for crop in CROPS_FAST:
        split_dir = (
            root
            / f"{crop}_leaf"
            / split
        )

        classes = sorted(
            [
                p.name
                for p in split_dir.iterdir()
                if p.is_dir()
            ]
        )

        crop_classes[crop] = classes

        disease_to_idx[crop] = {
            cls: idx
            for idx, cls in enumerate(classes)
        }

    for crop in CROPS_FAST:
        for cls in crop_classes[crop]:
            global_to_idx[
                (crop, cls)
            ] = len(
                global_classes
            )

            global_classes.append(
                f"{crop}::{cls}"
            )

    crop_to_idx = {
        crop: i
        for i, crop in enumerate(CROPS_FAST)
    }

    for crop in CROPS_FAST:
        crop_idx = crop_to_idx[
            crop
        ]

        split_dir = (
            root
            / f"{crop}_leaf"
            / split
        )

        for cls in crop_classes[crop]:
            disease_idx = disease_to_idx[
                crop
            ][
                cls
            ]

            global_idx = global_to_idx[
                (crop, cls)
            ]

            cls_dir = (
                split_dir
                / cls
            )

            for f in sorted(
                cls_dir.rglob("*")
            ):
                if not (
                    f.is_file()
                    and f.suffix.lower()
                    in IMAGE_EXTS_FAST
                ):
                    continue

                if (
                    split == "train"
                    and "__aug_"
                    in f.stem.lower()
                ):
                    continue

                samples.append(
                    (
                        str(f),
                        crop_idx,
                        disease_idx,
                        global_idx,
                        crop,
                        cls,
                    )
                )

    return samples


def _sample_hash_fast(samples):
    h = hashlib.sha256()

    for (
        path,
        crop_idx,
        disease_idx,
        global_idx,
        crop,
        cls,
    ) in samples:

        try:
            rel = (
                Path(path)
                .resolve()
                .relative_to(
                    FAST_DATA_ROOT.resolve()
                )
                .as_posix()
            )
        except Exception:
            rel = "/".join(
                Path(path)
                .as_posix()
                .split("/")[-4:]
            )

        h.update(
            (
                f"{rel}|{crop_idx}|{disease_idx}|"
                f"{global_idx}|{crop}|{cls}\n"
            ).encode(
                "utf-8"
            )
        )

    return h.hexdigest()


class _SpatialCacheDataset(Dataset):
    def __init__(
        self,
        samples,
        spatial_transform,
    ):
        self.samples = samples
        self.spatial_transform = spatial_transform

    def __len__(self):
        return len(
            self.samples
        )

    def __getitem__(
        self,
        idx,
    ):
        path = self.samples[
            idx
        ][
            0
        ]

        with Image.open(
            path
        ) as img:
            img = img.convert(
                "RGB"
            )

            # mean=0,std=1 => tensor in [0,1] after the exact same
            # deterministic resize/crop used by timm.
            x = self.spatial_transform(
                img
            )

        # Convert back to 8-bit AFTER deterministic spatial processing.
        # The worker later performs float conversion + model normalization
        # directly on the T4.
        x = (
            x.mul(
                255.0
            )
            .round()
            .clamp(
                0,
                255,
            )
            .to(
                torch.uint8
            )
        )

        return x


def _resolve_deit_fast():
    candidates = [
        "deit_tiny_patch16_224.fb_in1k",
        "deit_tiny_patch16_224",
    ]

    try:
        available = set(
            timm.list_models(
                include_tags=True
            )
        )
    except TypeError:
        available = set(
            timm.list_models()
        )

    for name in candidates:
        if name in available:
            return name

    matches = timm.list_models(
        "*deit_tiny_patch16_224*"
    )

    if not matches:
        raise RuntimeError(
            "Could not resolve DeiT-Tiny in timm."
        )

    return matches[
        0
    ]


def _cache_is_valid(
    cache_dir,
    expected_split_info,
):
    meta_path = (
        cache_dir
        / "cache_meta.json"
    )

    if not meta_path.exists():
        return False

    try:
        meta = json.loads(
            meta_path.read_text(
                encoding="utf-8"
            )
        )

        if meta.get(
            "backbone"
        ) != "deit_tiny":
            return False

        for split, info in expected_split_info.items():
            s = meta[
                "splits"
            ][
                split
            ]

            if int(
                s[
                    "count"
                ]
            ) != int(
                info[
                    "count"
                ]
            ):
                return False

            if (
                s[
                    "sample_hash"
                ]
                != info[
                    "sample_hash"
                ]
            ):
                return False

            arr_path = (
                cache_dir
                / f"{split}_images.npy"
            )

            if not arr_path.exists():
                return False

            arr = np.load(
                arr_path,
                mmap_mode="r",
            )

            if (
                tuple(
                    arr.shape
                )
                != (
                    info[
                        "count"
                    ],
                    3,
                    224,
                    224,
                )
                or arr.dtype
                != np.uint8
            ):
                return False

        return True

    except Exception:
        return False


def build_deit_uint8_cache():
    cache_dir = (
        SPEED_CACHE_ROOT
        / "deit_tiny"
    )

    samples_by_split = {
        split: _discover_samples_fast(
            FAST_DATA_ROOT,
            split,
        )
        for split in [
            "train",
            "val",
            "test",
        ]
    }

    expected_counts = {
        "train": 13720,
        "val": 3660,
        "test": 916,
    }

    split_info = {}

    for split, samples in samples_by_split.items():
        if len(
            samples
        ) != expected_counts[
            split
        ]:
            raise RuntimeError(
                f"Fast-cache dataset count mismatch for {split}: "
                f"{len(samples)} != {expected_counts[split]}"
            )

        split_info[
            split
        ] = {
            "count": len(
                samples
            ),
            "sample_hash": _sample_hash_fast(
                samples
            ),
        }

    if _cache_is_valid(
        cache_dir,
        split_info,
    ):
        print(
            "FAST CACHE ALREADY VALID:",
            cache_dir,
        )

        return cache_dir

    if cache_dir.exists():
        shutil.rmtree(
            cache_dir
        )

    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    model_name = _resolve_deit_fast()

    cfg_model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=0,
        global_pool="avg",
    )

    cfg = resolve_model_data_config(
        cfg_model
    )

    del cfg_model
    gc.collect()

    # Same deterministic spatial processing as the final no-augmentation run,
    # but without CPU normalization.
    spatial_tf = create_transform(
        input_size=(
            3,
            224,
            224,
        ),
        is_training=False,
        mean=(
            0.0,
            0.0,
            0.0,
        ),
        std=(
            1.0,
            1.0,
            1.0,
        ),
        interpolation=cfg.get(
            "interpolation",
            "bicubic",
        ),
        crop_pct=cfg.get(
            "crop_pct",
            0.875,
        ),
    )

    workers = max(
        1,
        min(
            4,
            os.cpu_count()
            or 4,
        ),
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        "BUILDING SHARED DeiT UINT8 FAST CACHE"
    )

    print(
        "This is a one-time cost; both DeiT models reuse the same cache."
    )

    print(
        "Workers:",
        workers,
    )

    print(
        "Model preprocessing:",
        model_name,
    )

    print(
        "=" * 100
    )

    meta = {
        "format":
            "LeafCareNet-uint8-spatial-cache-v1",

        "backbone":
            "deit_tiny",

        "resolved_model":
            model_name,

        "image_size":
            224,

        "mean":
            [
                float(x)
                for x
                in cfg[
                    "mean"
                ]
            ],

        "std":
            [
                float(x)
                for x
                in cfg[
                    "std"
                ]
            ],

        "interpolation":
            cfg.get(
                "interpolation",
                "bicubic",
            ),

        "crop_pct":
            float(
                cfg.get(
                    "crop_pct",
                    0.875,
                )
            ),

        "splits":
            {},
    }

    for split in [
        "train",
        "val",
        "test",
    ]:
        samples = samples_by_split[
            split
        ]

        ds = _SpatialCacheDataset(
            samples,
            spatial_tf,
        )

        loader = DataLoader(
            ds,
            batch_size=256,
            shuffle=False,
            num_workers=workers,
            pin_memory=False,
            persistent_workers=(
                workers > 0
            ),
            prefetch_factor=(
                3
                if workers > 0
                else None
            ),
        )

        out_path = (
            cache_dir
            / f"{split}_images.npy"
        )

        mmap = np.lib.format.open_memmap(
            out_path,
            mode="w+",
            dtype=np.uint8,
            shape=(
                len(
                    ds
                ),
                3,
                224,
                224,
            ),
        )

        offset = 0

        for batch in tqdm(
            loader,
            desc=f"cache {split}",
        ):
            n = int(
                batch.shape[
                    0
                ]
            )

            mmap[
                offset:
                offset + n
            ] = batch.numpy()

            offset += n

        mmap.flush()

        del mmap
        del loader
        del ds

        gc.collect()

        meta[
            "splits"
        ][
            split
        ] = {
            "count":
                len(
                    samples
                ),

            "sample_hash":
                split_info[
                    split
                ][
                    "sample_hash"
                ],

            "file":
                out_path.name,
        }

        print(
            f"Cached {split}: "
            f"{len(samples)} images | "
            f"{out_path.stat().st_size / (1024**3):.2f} GiB"
        )

    (
        cache_dir
        / "cache_meta.json"
    ).write_text(
        json.dumps(
            meta,
            indent=2,
        ),
        encoding="utf-8",
    )

    total_bytes = sum(
        p.stat().st_size
        for p
        in cache_dir.glob(
            "*.npy"
        )
    )

    print(
        "\nFAST CACHE READY:",
        cache_dir,
    )

    print(
        f"Cache size: "
        f"{total_bytes / (1024**3):.2f} GiB"
    )

    return cache_dir


if "deit_tiny" in FAST_CACHE_BACKBONES:
    DEIT_FAST_CACHE_DIR = build_deit_uint8_cache()
else:
    DEIT_FAST_CACHE_DIR = None
    print(
        "No pending DeiT experiment; fast DeiT cache not required."
    )


Fast-cache dataset root: /kaggle/input/datasets/mahbubur03078/leafdisease2/Final Dataset_Kaggle_FinalAugmented

BUILDING SHARED DeiT UINT8 FAST CACHE
This is a one-time cost; both DeiT models reuse the same cache.
Workers: 4
Model preprocessing: deit_tiny_patch16_224.fb_in1k


cache train:   0%|          | 0/54 [00:00<?, ?it/s]

Cached train: 13720 images | 1.92 GiB


cache val:   0%|          | 0/15 [00:00<?, ?it/s]

Cached val: 3660 images | 0.51 GiB


cache test:   0%|          | 0/4 [00:00<?, ?it/s]

Cached test: 916 images | 0.13 GiB

FAST CACHE READY: /kaggle/working/leafcarenet_fast_uint8_cache/deit_tiny
Cache size: 2.56 GiB


In [9]:

# ============================================================
# 4. DYNAMIC TWO-GPU TRAINING QUEUE + PER-MODEL FULL ZIP
# ============================================================


def stream_output(process, prefix, log_path):
    with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
        for line in process.stdout:
            log_file.write(line)
            stripped = line.rstrip()
            if stripped:
                print(f"[{prefix}] {stripped}", flush=True)


def write_master_state(active=None, failures=None):
    active = active or {}
    failures = failures or []
    state = {
        "updated_at": datetime.now().isoformat(),
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "workers_per_model": PER_MODEL_NUM_WORKERS,
        "online_augmentation": False,
        "dataset_counts": {"train": 13720, "val": 3660, "test": 916},
        "completed": [e for e in EXPERIMENT_ORDER if is_complete(e)],
        "pending": [e for e in EXPERIMENT_ORDER if not is_complete(e)],
        "active": active,
        "failures": failures,
        "model_registry": MODEL_REGISTRY,
    }
    MASTER_STATE.write_text(json.dumps(state, indent=2), encoding="utf-8")
    return state


def make_model_resume_zip(exp_name):
    """Create a full, independently restorable bundle immediately after completion."""
    root = model_root(exp_name)
    if not is_complete(exp_name):
        raise RuntimeError(f"Cannot zip incomplete experiment: {exp_name}")

    # Include exact training worker used in this run.
    shutil.copy2(
        WORKER_SCRIPT_PATH,
        root / "leafcarenet_journal8_worker.py",
    )

    manifest = {
        "bundle_format": "LeafCareNet-JOURNAL8-v1",
        "experiment": exp_name,
        "display_name": MODEL_META[exp_name]["display_name"],
        "role": MODEL_META[exp_name]["role"],
        "created_at": datetime.now().isoformat(),
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "dataset_counts": {"train": 13720, "val": 3660, "test": 916},
        "online_augmentation": False,
        "restore_instruction": (
            "Attach this ZIP and the clean dataset as Kaggle inputs, then run "
            "the same notebook. It will auto-restore and skip this model."
        ),
    }
    (root / "RESUME_BUNDLE_MANIFEST.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    zip_path = MODEL_ZIP_DIR / f"JOURNAL8_{exp_name}_COMPLETE.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=5,
    ) as zf:
        # Put model-root contents directly at ZIP root so restoration is simple.
        for file_path in root.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=str(file_path.relative_to(root)))

    with zipfile.ZipFile(zip_path, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"ZIP integrity failure: {bad}")

    print(
        f"BACKUP READY: {zip_path} "
        f"({zip_path.stat().st_size / (1024**2):.1f} MB)"
    )
    return zip_path


def make_failure_zip(exp_name, log_path):
    zip_path = MODEL_ZIP_DIR / f"JOURNAL8_{exp_name}_FAILED_DEBUG.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        if Path(log_path).exists():
            zf.write(log_path, arcname=Path(log_path).name)
        cfg = model_root(exp_name) / "experiment_config.json"
        if cfg.exists():
            zf.write(cfg, arcname="experiment_config.json")
        zf.write(WORKER_SCRIPT_PATH, arcname="leafcarenet_journal8_worker.py")
    return zip_path


def launch_worker(exp_name, physical_gpu):
    output_root = model_root(exp_name)
    output_root.mkdir(parents=True, exist_ok=True)

    log_path = BASE_OUTPUT / f"{exp_name}.console.log"

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
    env["LEAFCARENET_MODE"] = "train"
    env["LEAFCARENET_PREFLIGHT"] = "0"
    env["LEAFCARENET_EXPERIMENT"] = exp_name
    env["LEAFCARENET_OUTPUT_ROOT"] = str(output_root)
    env["LEAFCARENET_BATCH_SIZE"] = str(TRAIN_BATCH_SIZE)
    env["LEAFCARENET_EVAL_BATCH_SIZE"] = str(EVAL_BATCH_SIZE)
    env["LEAFCARENET_NUM_WORKERS"] = str(PER_MODEL_NUM_WORKERS)
    env["LEAFCARENET_PREFETCH_FACTOR"] = str(PREFETCH_FACTOR)

    # Shared deterministic preprocessing cache for the two pending DeiT jobs.
    # Both worker processes memory-map the SAME files; Linux page cache is shared.
    backbone = MODEL_META[
        exp_name
    ][
        "backbone"
    ]

    candidate_cache = (
        SPEED_CACHE_ROOT
        / backbone
    )

    if (
        candidate_cache
        / "cache_meta.json"
    ).exists():
        env[
            "LEAFCARENET_UINT8_CACHE_DIR"
        ] = str(
            candidate_cache
        )

        print(
            f"FAST CACHE -> {exp_name}: "
            f"{candidate_cache}"
        )

    process = subprocess.Popen(
        [sys.executable, "-u", str(WORKER_SCRIPT_PATH)],
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    thread = threading.Thread(
        target=stream_output,
        args=(process, f"GPU{physical_gpu} {exp_name}", log_path),
        daemon=True,
    )
    thread.start()

    print(
        f"STARTED GPU{physical_gpu}: {MODEL_META[exp_name]['display_name']} "
        f"(pid={process.pid})"
    )

    return {
        "experiment": exp_name,
        "gpu": physical_gpu,
        "process": process,
        "thread": thread,
        "log_path": log_path,
        "started_at": time.time(),
    }


# Any model restored from an OLD legacy ZIP is now canonicalized.
# Before training starts, convert every verified completed model into the
# new JOURNAL8_*_COMPLETE.zip format so future sessions need only the
# standardized bundles.
print("\nStandardizing every verified completed model into JOURNAL8 resume ZIPs...")

# If current-session/restored outputs completed before this cell is rerun,
# ensure each has a standardized ZIP.
for exp_name in EXPERIMENT_ORDER:
    if is_complete(exp_name):
        target_zip = MODEL_ZIP_DIR / f"JOURNAL8_{exp_name}_COMPLETE.zip"
        if not target_zip.exists():
            make_model_resume_zip(exp_name)

pending = [e for e in EXPERIMENT_ORDER if not is_complete(e)]
print("\nPending queue:", pending)

if not pending:
    print("All eight experiments are already complete.")
else:
    # Kaggle T4x2 is required for the requested dynamic two-GPU queue.
    try:
        import torch
        gpu_count = torch.cuda.device_count()
    except Exception:
        gpu_count = 0

    if gpu_count < 2:
        raise RuntimeError(
            f"This notebook requires 2 visible GPUs; found {gpu_count}. "
            "Select Kaggle GPU T4 x2."
        )

    queue = list(pending)
    active = {}
    failures = []
    allow_new_launches = True

    # Fill both GPUs initially.
    for gpu in [0, 1]:
        if queue:
            exp_name = queue.pop(0)
            active[gpu] = launch_worker(exp_name, gpu)

    write_master_state(
        active={g: j["experiment"] for g, j in active.items()},
        failures=failures,
    )

    # Dynamic scheduler: a free GPU receives the next model immediately.
    while active:
        time.sleep(2.0)

        finished_gpus = []

        for gpu, job in list(active.items()):
            rc = job["process"].poll()
            if rc is None:
                continue

            job["thread"].join()
            elapsed = time.time() - job["started_at"]
            exp_name = job["experiment"]

            print(
                f"\nFINISHED GPU{gpu}: {exp_name} | rc={rc} | "
                f"wall={elapsed/60:.1f} min"
            )

            if rc == 0 and is_complete(exp_name):
                make_model_resume_zip(exp_name)
            else:
                failure_zip = make_failure_zip(exp_name, job["log_path"])
                failures.append(
                    {
                        "experiment": exp_name,
                        "gpu": gpu,
                        "return_code": rc,
                        "log": str(job["log_path"]),
                        "debug_zip": str(failure_zip),
                    }
                )
                allow_new_launches = False
                print("A model failed; no new jobs will be launched until this is fixed.")

            finished_gpus.append(gpu)

        for gpu in finished_gpus:
            del active[gpu]

        # Launch next pending model on every newly-free GPU, without waiting
        # for the other GPU's model to finish.
        if allow_new_launches:
            for gpu in [0, 1]:
                if gpu not in active and queue:
                    exp_name = queue.pop(0)
                    active[gpu] = launch_worker(exp_name, gpu)

        write_master_state(
            active={g: j["experiment"] for g, j in active.items()},
            failures=failures,
        )

    write_master_state(active={}, failures=failures)

    if failures:
        print(json.dumps(failures, indent=2))
        raise RuntimeError(
            "At least one experiment failed. Completed models were preserved "
            "and zipped; fix the failed model and rerun this cell to continue."
        )

print("\nDYNAMIC TRAINING QUEUE COMPLETE")
print("Per-model ZIP directory:", MODEL_ZIP_DIR)


# Free the temporary ~2.6 GiB execution cache after all model bundles exist.
# It is not needed for analysis or future resume once both DeiT models are complete.
if all(
    is_complete(
        exp_name
    )
    for exp_name
    in EXPERIMENT_ORDER
):
    if SPEED_CACHE_ROOT.exists():
        try:
            shutil.rmtree(
                SPEED_CACHE_ROOT
            )

            print(
                "Removed temporary fast preprocessing cache:",
                SPEED_CACHE_ROOT,
            )

        except Exception as exc:
            print(
                "Fast-cache cleanup warning:",
                repr(
                    exc
                ),
            )



Standardizing every verified completed model into JOURNAL8 resume ZIPs...

Pending queue: ['04_flat_deit_tiny', '08_hier_deit_tiny']
FAST CACHE -> 04_flat_deit_tiny: /kaggle/working/leafcarenet_fast_uint8_cache/deit_tiny
STARTED GPU0: DeiT-Tiny Flat (pid=181)
FAST CACHE -> 08_hier_deit_tiny: /kaggle/working/leafcarenet_fast_uint8_cache/deit_tiny
STARTED GPU1: DeiT-Tiny Hierarchical (pid=183)
[GPU1 08_hier_deit_tiny] ====================================================================================================
[GPU0 04_flat_deit_tiny] ====================================================================================================
[GPU0 04_flat_deit_tiny] FINAL JOURNAL RUN LOG STARTED: 2026-09-20 16:37:43
[GPU1 08_hier_deit_tiny] FINAL JOURNAL RUN LOG STARTED: 2026-09-20 16:37:43
[GPU1 08_hier_deit_tiny] Log file: /kaggle/working/leafcarenet_journal8_dynamic/08_hier_deit_tiny/logs/training_run.log
[GPU0 04_flat_deit_tiny] Log file: /kaggle/working/leafcarenet_journal8_dynamic/

In [ ]:
# ============================================================
# FORCE ALL 8 MODELS' CONFUSION MATRICES TO JOURNAL BLUE
# Shortened Class Names + 45° Angle + Large Bold Center Numbers
# ============================================================

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

REPORT_SUMMARY_KEYS = {
    "accuracy", "macro avg", "weighted avg", "micro avg", "samples avg",
}

SHORT_CLASS_MAPPING = {
    'potato::Potato_Early_blight': 'Potato EB',
    'potato::Potato_Late_blight': 'Potato LB',
    'potato::Potato_healthy': 'Potato Healthy',
    'rice::Rice_Bacterial_Leaf_Blight': 'Rice BLB',
    'rice::Rice_Brown_Spot': 'Rice Brown Spot',
    'rice::Rice_Healthy_Leaf': 'Rice Healthy',
    'rice::Rice_Leaf_Blast': 'Rice Blast',
    'rice::Rice_Sheath_Blight': 'Rice Sheath Blight',
    'tomato::Tomato_Bacterial_spot': 'Tomato Bact. Spot',
    'tomato::Tomato_Early_blight': 'Tomato EB',
    'tomato::Tomato_Late_blight': 'Tomato LB',
    'tomato::Tomato_Leaf_Mold': 'Tomato Leaf Mold',
    'tomato::Tomato_Septoria_leaf_spot': 'Tomato Septoria',
    'tomato::Tomato_Spider_mites_Two_spotted_spider_mite': 'Tomato Spider Mite',
    'tomato::Tomato_Target_Spot': 'Tomato Target Spot',
    'tomato::Tomato_Tomato_YellowLeaf_Curl_Virus': 'Tomato TYLCV',
    'tomato::Tomato_Tomato_mosaic_virus': 'Tomato TMV',
    'tomato::Tomato_healthy': 'Tomato Healthy',
    'potato': 'Potato', 'rice': 'Rice', 'tomato': 'Tomato',
}

def _get_short_name(label):
    if label in SHORT_CLASS_MAPPING:
        return SHORT_CLASS_MAPPING[label]
    clean = label.replace('tomato::', 'Tomato ').replace('rice::', 'Rice ').replace('potato::', 'Potato ')
    clean = clean.replace('Tomato___', 'Tomato ').replace('Rice___', 'Rice ').replace('Potato___', 'Potato ')
    clean = clean.replace('_', ' ').strip()
    return clean

def _labels_from_report(report_path):
    report = json.loads(Path(report_path).read_text(encoding="utf-8"))
    return [str(k) for k in report.keys() if k not in REPORT_SUMMARY_KEYS]

def _plot_blue_cm(cm, labels, title, out_path, normalize=False):
    cm = np.asarray(cm)
    n = len(labels)
    short_labels = [_get_short_name(l) for l in labels]
    
    if normalize:
        cm_show = cm.astype(float)
        row_sums = cm_show.sum(axis=1, keepdims=True)
        cm_show = np.divide(cm_show, row_sums, out=np.zeros_like(cm_show, dtype=float), where=row_sums != 0) * 100.0
    else:
        cm_show = cm

    if n <= 3:
        figsize, num_fontsize, label_fontsize = (6.5, 5.6), 13.0, 10.5
    elif n <= 5:
        figsize, num_fontsize, label_fontsize = (7.8, 6.8), 11.5, 10.0
    elif n <= 8:
        figsize, num_fontsize, label_fontsize = (9.2, 8.0), 10.5, 9.5
    else:
        figsize, num_fontsize, label_fontsize = (11.5, 9.8), 9.5, 9.0

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    
    annot_data = np.empty_like(cm_show, dtype=object)
    for i in range(cm_show.shape[0]):
        for j in range(cm_show.shape[1]):
            val = cm_show[i, j]
            if normalize:
                if val >= 1.0:
                    annot_data[i, j] = f"{val:.0f}%" if val == 100.0 or val >= 99.5 else f"{val:.1f}%"
                elif val > 0.0:
                    annot_data[i, j] = f"{val:.1f}%"
                else:
                    annot_data[i, j] = ""
            else:
                annot_data[i, j] = f"{int(val)}" if val > 0 else "0"

    sns.heatmap(cm_show, annot=annot_data, fmt="", cmap="Blues",
                vmin=0, vmax=100.0 if normalize else None,
                ax=ax, cbar=True,
                cbar_kws={'label': 'Normalized Accuracy [%]' if normalize else 'Sample Count'},
                xticklabels=short_labels, yticklabels=short_labels,
                linewidths=0.3, linecolor='#B0BEC5',
                annot_kws={'fontsize': num_fontsize, 'fontweight': 'bold'})

    ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=label_fontsize, fontweight='bold')
    ax.set_yticklabels(short_labels, rotation=0, fontsize=label_fontsize, fontweight='bold')
    ax.set_xlabel("Predicted Class", fontsize=label_fontsize + 1.5, fontweight='bold', labelpad=10)
    ax.set_ylabel("True Class", fontsize=label_fontsize + 1.5, fontweight='bold', labelpad=10)
    if title:
        ax.set_title(title, fontsize=label_fontsize + 2.0, fontweight='bold', pad=12)
        
    plt.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

print("[+] Cell 6 updated with short names, 45° rotation, and large bold numbers.")



In [10]:

# ============================================================
# 5. FINAL JOURNAL TABLES, STATISTICS, AND COMPARISON FIGURES
# ============================================================

import matplotlib.pyplot as plt
from scipy.stats import binomtest

COMPARISON_ROOT = BASE_OUTPUT / "FINAL_COMPARISON"
COMPARISON_FIG = COMPARISON_ROOT / "figures_300dpi"
COMPARISON_TABLE = COMPARISON_ROOT / "tables"
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_FIG.mkdir(parents=True, exist_ok=True)
COMPARISON_TABLE.mkdir(parents=True, exist_ok=True)

missing = [e for e in EXPERIMENT_ORDER if not is_complete(e)]
if missing:
    raise RuntimeError(
        "Final comparison requires all 8 models. Missing: " + ", ".join(missing)
    )

# ------------------------------------------------------------
# Merge model results
# ------------------------------------------------------------
rows = []
for exp_name in EXPERIMENT_ORDER:
    result = json.loads(result_file(exp_name).read_text(encoding="utf-8"))
    meta = MODEL_META[exp_name]
    row = dict(result)
    row["display_name"] = meta["display_name"]
    row["role"] = meta["role"]
    row["backbone_group"] = meta["backbone"]
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_csv = COMPARISON_TABLE / "FINAL_8MODEL_COMPARISON.csv"
comparison_df.to_csv(comparison_csv, index=False)

journal_cols = [
    "display_name", "role", "test_accuracy", "test_macro_f1",
    "test_weighted_f1", "test_balanced_accuracy", "test_mcc",
    "test_cohen_kappa", "accuracy_ci95_low", "accuracy_ci95_high",
    "best_epoch", "training_seconds", "parameters", "checkpoint_mb",
    "t4_batch1_latency_mean_ms", "t4_batch1_latency_p95_ms",
    "t4_batch1_throughput_fps",
]
journal_cols = [c for c in journal_cols if c in comparison_df.columns]
comparison_df[journal_cols].to_csv(
    COMPARISON_TABLE / "JOURNAL_MAIN_COMPARISON_TABLE.csv", index=False
)

print("\nFINAL 8-MODEL COMPARISON")
print(comparison_df[journal_cols].to_string(index=False))

names = comparison_df["display_name"].tolist()
x = np.arange(len(names))

# ------------------------------------------------------------
# Figure 1: core predictive metrics
# ------------------------------------------------------------
core_metrics = [
    ("test_accuracy", "Accuracy"),
    ("test_macro_f1", "Macro-F1"),
    ("test_weighted_f1", "Weighted-F1"),
    ("test_balanced_accuracy", "Balanced Accuracy"),
]
width = 0.19
plt.figure(figsize=(16, 7))
for k, (col, label) in enumerate(core_metrics):
    plt.bar(x + (k - 1.5) * width, comparison_df[col], width=width, label=label)
plt.xticks(x, names, rotation=32, ha="right")
plt.ylim(0.90, 1.01)
plt.ylabel("Score")
plt.title("Predictive Performance Comparison")
plt.legend(ncol=4)
plt.tight_layout()
plt.savefig(COMPARISON_FIG / "01_predictive_metrics_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Figure 2: accuracy with bootstrap 95% CI
# ------------------------------------------------------------
y = comparison_df["test_accuracy"].to_numpy(float)
lo = comparison_df["accuracy_ci95_low"].to_numpy(float)
hi = comparison_df["accuracy_ci95_high"].to_numpy(float)
yerr = np.vstack([y - lo, hi - y])
plt.figure(figsize=(14, 6))
plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=4)
plt.xticks(x, names, rotation=32, ha="right")
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy with 95% Bootstrap Confidence Intervals")
plt.tight_layout()
plt.savefig(COMPARISON_FIG / "02_accuracy_bootstrap_ci.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Figures 3-7: efficiency
# ------------------------------------------------------------
def simple_bar(values, ylabel, title, filename):
    plt.figure(figsize=(14, 6))
    plt.bar(x, values)
    plt.xticks(x, names, rotation=32, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(COMPARISON_FIG / filename, dpi=300, bbox_inches="tight")
    plt.close()

simple_bar(
    comparison_df["parameters"] / 1e6,
    "Parameters (millions)",
    "Model Parameter Count",
    "03_parameter_count.png",
)
simple_bar(
    comparison_df["checkpoint_mb"],
    "Checkpoint Size (MB)",
    "Checkpoint Size Comparison",
    "04_checkpoint_size.png",
)
simple_bar(
    comparison_df["t4_batch1_latency_mean_ms"],
    "Batch-1 Latency on T4 (ms)",
    "Single-Image T4 Inference Latency",
    "05_t4_latency.png",
)
simple_bar(
    comparison_df["t4_batch1_throughput_fps"],
    "Throughput (images/s)",
    "T4 Batch-1 Throughput",
    "06_t4_throughput.png",
)
simple_bar(
    comparison_df["training_seconds"] / 60.0,
    "Training Time (min)",
    "Per-Model Training Time",
    "07_training_time.png",
)

# ------------------------------------------------------------
# Figures 8-10: performance-efficiency trade-offs
# ------------------------------------------------------------
def tradeoff_plot(xcol, ycol, xlabel, ylabel, title, filename, xscale=None):
    plt.figure(figsize=(10, 7))
    xv = comparison_df[xcol].to_numpy(float)
    yv = comparison_df[ycol].to_numpy(float)
    plt.scatter(xv, yv, s=70)
    for xx, yy, name in zip(xv, yv, names):
        plt.annotate(name, (xx, yy), xytext=(5, 5), textcoords="offset points", fontsize=8)
    if xscale:
        plt.xscale(xscale)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(COMPARISON_FIG / filename, dpi=300, bbox_inches="tight")
    plt.close()

comparison_df["parameters_m"] = comparison_df["parameters"] / 1e6
tradeoff_plot(
    "parameters_m", "test_accuracy",
    "Parameters (millions)", "Test Accuracy",
    "Accuracy vs Model Complexity", "08_accuracy_vs_parameters.png", xscale="log"
)
tradeoff_plot(
    "t4_batch1_latency_mean_ms", "test_accuracy",
    "T4 Batch-1 Latency (ms)", "Test Accuracy",
    "Accuracy vs Inference Latency", "09_accuracy_vs_latency.png"
)
tradeoff_plot(
    "parameters_m", "test_macro_f1",
    "Parameters (millions)", "Macro-F1",
    "Macro-F1 vs Model Complexity", "10_macrof1_vs_parameters.png", xscale="log"
)

# ------------------------------------------------------------
# Figure 11: hierarchy gain by backbone
# ------------------------------------------------------------
pairs = [
    ("MobileNetV2", "01_flat_mobilenetv2", "05_leafcarenet"),
    ("EfficientNet-B3", "02_flat_efficientnet_b3", "06_hier_efficientnet_b3"),
    ("FasterNet-M", "03_flat_fasternet_m", "07_hier_fasternet_m"),
    ("DeiT-Tiny", "04_flat_deit_tiny", "08_hier_deit_tiny"),
]
by_exp = comparison_df.set_index("experiment")
gain_rows = []
for backbone, flat_exp, hier_exp in pairs:
    gain_rows.append({
        "backbone": backbone,
        "accuracy_gain_pp": 100 * (
            by_exp.loc[hier_exp, "test_accuracy"] - by_exp.loc[flat_exp, "test_accuracy"]
        ),
        "macro_f1_gain_pp": 100 * (
            by_exp.loc[hier_exp, "test_macro_f1"] - by_exp.loc[flat_exp, "test_macro_f1"]
        ),
    })
gain_df = pd.DataFrame(gain_rows)
gain_df.to_csv(COMPARISON_TABLE / "HIERARCHY_GAIN_BY_BACKBONE.csv", index=False)

xx = np.arange(len(gain_df))
width = 0.36
plt.figure(figsize=(10, 6))
plt.bar(xx - width/2, gain_df["accuracy_gain_pp"], width=width, label="Accuracy gain")
plt.bar(xx + width/2, gain_df["macro_f1_gain_pp"], width=width, label="Macro-F1 gain")
plt.axhline(0, linewidth=1)
plt.xticks(xx, gain_df["backbone"])
plt.ylabel("Hierarchical − Flat (percentage points)")
plt.title("Effect of Hierarchical Classification Across Backbones")
plt.legend()
plt.tight_layout()
plt.savefig(COMPARISON_FIG / "11_hierarchy_gain_by_backbone.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Figure 12: hierarchical stage-wise performance
# ------------------------------------------------------------
hier_df = comparison_df[comparison_df["family"] == "hierarchical"].copy()
stage_cols = [
    ("test_stage1_accuracy", "Crop Router"),
    ("test_potato_oracle_accuracy", "Potato Oracle"),
    ("test_rice_oracle_accuracy", "Rice Oracle"),
    ("test_tomato_oracle_accuracy", "Tomato Oracle"),
    ("test_accuracy", "True End-to-End"),
]
hx = np.arange(len(hier_df))
hw = 0.16
plt.figure(figsize=(13, 7))
for k, (col, label) in enumerate(stage_cols):
    plt.bar(hx + (k - 2) * hw, hier_df[col], width=hw, label=label)
plt.xticks(hx, hier_df["display_name"], rotation=25, ha="right")
plt.ylim(0.90, 1.01)
plt.ylabel("Accuracy")
plt.title("Hierarchical Stage-Wise Test Performance")
plt.legend(ncol=3)
plt.tight_layout()
plt.savefig(COMPARISON_FIG / "12_hierarchical_stage_performance.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Exact McNemar tests: every model vs proposed LeafCareNet
# ------------------------------------------------------------
def prediction_csv(exp_name):
    return model_root(exp_name) / "predictions" / f"{exp_name}_test_predictions.csv"

leaf_exp = "05_leafcarenet"
leaf_pred = pd.read_csv(prediction_csv(leaf_exp)).sort_values("sample_index").reset_index(drop=True)

mcnemar_rows = []
for exp_name in EXPERIMENT_ORDER:
    if exp_name == leaf_exp:
        continue
    pred = pd.read_csv(prediction_csv(exp_name)).sort_values("sample_index").reset_index(drop=True)

    if len(pred) != len(leaf_pred):
        raise RuntimeError(f"Prediction count mismatch: {exp_name}")

    def stable_test_key(path_value):
        # Ignore session-specific /kaggle/input prefixes.
        # The final three path components are normally:
        # test / class / filename.
        parts = str(path_value).replace("\\", "/").split("/")
        parts = [p for p in parts if p]
        return "/".join(parts[-3:])

    pred_keys = pred["path"].astype(str).map(stable_test_key).to_numpy()
    leaf_keys = leaf_pred["path"].astype(str).map(stable_test_key).to_numpy()

    if not (pred_keys == leaf_keys).all():
        raise RuntimeError(
            f"Prediction test-image/order mismatch after stable path normalization: {exp_name}"
        )

    a = pred["correct"].astype(bool).to_numpy()
    b_leaf = leaf_pred["correct"].astype(bool).to_numpy()
    baseline_correct_leaf_wrong = int(np.sum(a & ~b_leaf))
    baseline_wrong_leaf_correct = int(np.sum(~a & b_leaf))
    discordant = baseline_correct_leaf_wrong + baseline_wrong_leaf_correct
    p = 1.0 if discordant == 0 else float(
        binomtest(
            min(baseline_correct_leaf_wrong, baseline_wrong_leaf_correct),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )

    mcnemar_rows.append({
        "comparison_model": MODEL_META[exp_name]["display_name"],
        "comparison_experiment": exp_name,
        "leafcarenet": "LeafCareNet (Proposed)",
        "model_correct_leaf_wrong_b": baseline_correct_leaf_wrong,
        "model_wrong_leaf_correct_c": baseline_wrong_leaf_correct,
        "discordant_pairs": discordant,
        "exact_mcnemar_p": p,
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(COMPARISON_TABLE / "MCNEMAR_ALL_VS_LEAFCARENET.csv", index=False)
print("\nExact McNemar tests vs LeafCareNet:")
print(mcnemar_df.to_string(index=False))

# ------------------------------------------------------------
# Create small journal-analysis ZIP (no checkpoints) and final state
# ------------------------------------------------------------
write_master_state(active={}, failures=[])

analysis_zip = Path("/kaggle/working/LeafCareNet_JOURNAL8_ANALYSIS_RESULTS.zip")
if analysis_zip.exists():
    analysis_zip.unlink()

with zipfile.ZipFile(analysis_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    # Final comparison outputs
    for p in COMPARISON_ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(Path("FINAL_COMPARISON") / p.relative_to(COMPARISON_ROOT)))

    # Per-model result/report/prediction/figure/log files, excluding checkpoints.
    for exp_name in EXPERIMENT_ORDER:
        root = model_root(exp_name)
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            if "checkpoints" in p.parts:
                continue
            zf.write(p, arcname=str(Path(exp_name) / p.relative_to(root)))

    zf.write(MASTER_STATE, arcname="MASTER_STATE.json")
    zf.write(WORKER_SCRIPT_PATH, arcname="leafcarenet_journal8_worker.py")

with zipfile.ZipFile(analysis_zip, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"Analysis ZIP integrity failure: {bad}")

print("\nJournal analysis ZIP:", analysis_zip)
print("Per-model full resume ZIPs:", MODEL_ZIP_DIR)
print("Comparison CSV:", comparison_csv)



FINAL 8-MODEL COMPARISON
                display_name                            role  test_accuracy  test_macro_f1  test_weighted_f1  test_balanced_accuracy  test_mcc  test_cohen_kappa  accuracy_ci95_low  accuracy_ci95_high  best_epoch  training_seconds  parameters  checkpoint_mb  t4_batch1_latency_mean_ms  t4_batch1_latency_p95_ms  t4_batch1_throughput_fps
            MobileNetV2 Flat     Same-backbone flat baseline       0.990175       0.985279          0.990215                0.984659  0.989039          0.989024           0.983624            0.995633          12       3319.994337     2244368       8.806773                   5.244738                  5.616028                190.667276
        EfficientNet-B3 Flat                    CNN baseline       0.990175       0.982787          0.990140                0.984465  0.989042          0.989024           0.984716            0.995633           9       2957.094294    10720824      41.441978                  15.024856                 21

In [11]:

# ============================================================
# 6. BACKUP / DOWNLOAD CHECKLIST
# ============================================================

print("=" * 110)
print("FILES TO SAVE BEFORE STOPPING THE KAGGLE SESSION")
print("=" * 110)

for exp_name in EXPERIMENT_ORDER:
    z = MODEL_ZIP_DIR / f"JOURNAL8_{exp_name}_COMPLETE.zip"
    status = "READY" if z.exists() else "MISSING"
    size = f"{z.stat().st_size / (1024**2):.1f} MB" if z.exists() else "-"
    print(f"{status:7s} | {size:>10s} | {z}")

analysis_zip = Path("/kaggle/working/LeafCareNet_JOURNAL8_ANALYSIS_RESULTS.zip")
print("\nAnalysis-only ZIP:", analysis_zip, "| exists =", analysis_zip.exists())
print("\nFor a new Kaggle session:")
print("  1) attach the clean leafdisease2 dataset")
print("  2) attach every JOURNAL8_*_COMPLETE.zip already finished")
print("  3) run this notebook from the top")
print("  4) completed models are restored/skipped; the dynamic queue starts at the next pending model")
print("=" * 110)


print("\nLEGACY ZIP REUSE NOTE:")
print("  - Old ZIP filenames can be anything (e.g., '5 model.zip').")
print("  - The notebook reads internal result.json files and validates each model.")
print("  - Previous linear Hierarchical MobileNetV2 is mapped to LeafCareNet.")
print("  - Old FasterNet+SE and old FasterNet-based LeafCareNet are ignored.")
print("  - Check /kaggle/working/JOURNAL8_RESUME_AUDIT.csv to see exactly what was skipped/trained.")


FILES TO SAVE BEFORE STOPPING THE KAGGLE SESSION
READY   |     9.4 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_01_flat_mobilenetv2_COMPLETE.zip
READY   |    39.7 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_02_flat_efficientnet_b3_COMPLETE.zip
READY   |   187.1 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_03_flat_fasternet_m_COMPLETE.zip
READY   |    20.9 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_04_flat_deit_tiny_COMPLETE.zip
READY   |    11.6 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_05_leafcarenet_COMPLETE.zip
READY   |    41.7 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_06_hier_efficientnet_b3_COMPLETE.zip
READY   |   189.2 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_07_hier_fasternet_m_COMPLETE.zip
READY   |    22.9 MB | /kaggle/working/leafcarenet_journal8_model_zips/JOURNAL8_08_hier_deit_tiny_COMPLETE.zip

Analysis-only ZIP: /kaggle/working/LeafCareNet_

In [13]:

# ============================================================
# 7. ONE MASTER ZIP: ALL 8 MODELS + CHECKPOINTS + JOURNAL OUTPUTS
# ============================================================

from IPython.display import display, FileLink, Javascript

MASTER_ALL_ZIP = Path(
    "/kaggle/working/"
    "LeafCareNet_JOURNAL8_ALL_8_MODELS_COMPLETE.zip"
)

analysis_zip = Path(
    "/kaggle/working/"
    "LeafCareNet_JOURNAL8_ANALYSIS_RESULTS.zip"
)

# Make sure every verified model has a standardized full resume bundle.
missing = []

for exp_name in EXPERIMENT_ORDER:
    if not is_complete(
        exp_name
    ):
        missing.append(
            exp_name
        )
        continue

    model_zip = (
        MODEL_ZIP_DIR
        / f"JOURNAL8_{exp_name}_COMPLETE.zip"
    )

    if not model_zip.exists():
        make_model_resume_zip(
            exp_name
        )

if missing:
    raise RuntimeError(
        "Cannot create final 8-model master ZIP. "
        "Incomplete experiments: "
        + ", ".join(
            missing
        )
    )

model_zip_paths = [
    MODEL_ZIP_DIR
    / f"JOURNAL8_{exp_name}_COMPLETE.zip"
    for exp_name
    in EXPERIMENT_ORDER
]

for p in model_zip_paths:
    if not p.exists():
        raise FileNotFoundError(
            p
        )

if not analysis_zip.exists():
    raise FileNotFoundError(
        "Final analysis ZIP was not created. "
        "Run the comparison/statistics cell first."
    )

readme_path = Path(
    "/kaggle/working/"
    "JOURNAL8_MASTER_README.txt"
)

readme_path.write_text(
    """LeafCareNet final 8-model journal benchmark master archive

Contents
========
MODELS/
    Eight independently restorable JOURNAL8_*_COMPLETE.zip files.
    Each model ZIP contains checkpoint(s), predictions, result.json,
    classification reports, confusion matrices, training histories,
    figures, logs, configuration, completion metadata, and worker code.

ANALYSIS/
    LeafCareNet_JOURNAL8_ANALYSIS_RESULTS.zip
    Final eight-model comparison tables, 300-DPI journal graphs,
    McNemar tests, and non-checkpoint analysis outputs.

FINAL_COMPARISON/
    Direct copy of the final comparison figures/tables for convenience.

RESUME/
    Resume audit, master state, and the exact worker code.

Future Kaggle resume
====================
Attach the clean leafdisease2 dataset and the required model ZIP files from
MODELS/ as Kaggle inputs, then run the final notebook. Completed models are
validated and skipped.
""",
    encoding="utf-8",
)

if MASTER_ALL_ZIP.exists():
    MASTER_ALL_ZIP.unlink()

# The per-model files are already ZIP-compressed, so ZIP_STORED is faster and
# avoids wasting CPU time recompressing hundreds of MB of nested archives.
with zipfile.ZipFile(
    MASTER_ALL_ZIP,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as zf:

    # Eight full model/resume archives.
    for model_zip in model_zip_paths:
        zf.write(
            model_zip,
            arcname=str(
                Path(
                    "MODELS"
                )
                / model_zip.name
            ),
        )

    # Final analysis archive.
    zf.write(
        analysis_zip,
        arcname=str(
            Path(
                "ANALYSIS"
            )
            / analysis_zip.name
        ),
    )

    # Direct comparison outputs for immediate paper use.
    comparison_root = Path(
        "/kaggle/working/"
        "leafcarenet_journal8_final_comparison"
    )

    if comparison_root.exists():
        for p in comparison_root.rglob(
            "*"
        ):
            if p.is_file():
                zf.write(
                    p,
                    arcname=str(
                        Path(
                            "FINAL_COMPARISON"
                        )
                        / p.relative_to(
                            comparison_root
                        )
                    ),
                )

    # Resume / reproducibility metadata.
    audit = Path(
        "/kaggle/working/"
        "JOURNAL8_RESUME_AUDIT.csv"
    )

    if audit.exists():
        zf.write(
            audit,
            arcname=(
                "RESUME/"
                "JOURNAL8_RESUME_AUDIT.csv"
            ),
        )

    if MASTER_STATE.exists():
        zf.write(
            MASTER_STATE,
            arcname=(
                "RESUME/"
                "MASTER_STATE.json"
            ),
        )

    if WORKER_SCRIPT_PATH.exists():
        zf.write(
            WORKER_SCRIPT_PATH,
            arcname=(
                "RESUME/"
                "leafcarenet_journal8_worker.py"
            ),
        )

    zf.write(
        readme_path,
        arcname="README.txt",
    )

# Integrity check.
with zipfile.ZipFile(
    MASTER_ALL_ZIP,
    "r",
) as zf:
    bad = zf.testzip()

if bad is not None:
    raise RuntimeError(
        f"MASTER ZIP integrity failure: {bad}"
    )

size_gib = (
    MASTER_ALL_ZIP.stat().st_size
    / (
        1024 ** 3
    )
)

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL 8-MODEL MASTER ARCHIVE READY"
)

print(
    "=" * 110
)

print(
    "File:",
    MASTER_ALL_ZIP,
)

print(
    f"Size: {size_gib:.2f} GiB"
)

print(
    "Contains: all 8 full model ZIPs + checkpoints + predictions + "
    "reports + graphs + final comparison + statistics + resume metadata."
)

print(
    "=" * 110
)

# Clickable fallback link.
os.chdir(
    "/kaggle/working"
)

display(
    FileLink(
        MASTER_ALL_ZIP.name,
        result_html_prefix=(
            "<b>Download complete 8-model master ZIP:</b> "
        ),
    )
)

# Browser auto-download attempt. Kaggle/Chrome may block automatic downloads;
# the clickable link above remains available in that case.
display(
    Javascript(
        f"""
        setTimeout(function() {{
            var a = document.createElement("a");
            a.href = "files/{MASTER_ALL_ZIP.name}";
            a.download = "{MASTER_ALL_ZIP.name}";
            a.style.display = "none";
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        }}, 1200);
        """
    )
)

print(
    "Automatic browser download requested. "
    "If the browser blocks it, click the download link shown above."
)



FINAL 8-MODEL MASTER ARCHIVE READY
File: /kaggle/working/LeafCareNet_JOURNAL8_ALL_8_MODELS_COMPLETE.zip
Size: 0.53 GiB
Contains: all 8 full model ZIPs + checkpoints + predictions + reports + graphs + final comparison + statistics + resume metadata.


/kaggle/working/LeafCareNet_JOURNAL8_ALL_8_MODELS_COMPLETE.zip

<IPython.core.display.Javascript object>

Automatic browser download requested. If the browser blocks it, click the download link shown above.
